# Question 1 - Data Understanding

The objective of this notebook is to understand the available data,
identify the most relevant datasets for predicting the final regular
season ranking, and inspect their structure, quality and temporal coverage.

In [56]:
import pandas as pd

# Load the Teams dataset into a pandas DataFrame
df = pd.read_csv("../hockey_development_data/Teams.csv")

# Remove leading and trailing spaces from all column names
df.columns = df.columns.str.strip()

# Display the first 5 rows of the dataset
#df.head()

# Show the number of rows and columns in the dataset
#df.shape

# Display the names of all columns
#df.columns.tolist()

# Show information about the dataset, including column types and non-null values
#df.info()

# Count the number of missing values in each column
#df.isna().sum()

# Display descriptive statistics for all columns
#df.describe(include="all")

# Show the earliest and latest seasons available in the dataset
#df["year"].min(), df["year"].max()

# Count the number of unique teams in each season
#df.groupby("year")["tmID"].nunique()

# Check if there are duplicate entries for the same team in the same season
#df.duplicated(subset=["year", "tmID"]).sum()

### Data quality observation

Several column names contained leading or trailing whitespace.
The column names were stripped to ensure consistent access and avoid errors
during data manipulation.

### Initial observation

The number of teams varies across seasons. Early seasons contain fewer teams,
while the most recent seasons contain 30 teams.

This variation may need to be considered later when defining the training period.

In [57]:
# Show team name and final rank for the first season
df[df["year"] == 1909][["tmID", "name", "rank"]].sort_values("rank")

,tmID,name,rank
4,MOW,Montreal Wanderers,1
5,OT1,Ottawa Senators,2
6,REN,Renfrew Creamery Kings,3
0,COB,Cobalt Silver Kings,4
1,HAI,Haileybury Hockey Club,5
3,MOS,Montreal Shamrocks,6
2,LES,Les Canadiens,7


In [58]:
# Show team name and final rank for the most recent season
df[df["year"] == df["year"].max()][
    ["tmID", "name", "rank"]
].sort_values("rank")

,tmID,name,rank
1458,WAS,Washington Capitals,1
1446,NJD,New Jersey Devils,1
1432,BUF,Buffalo Sabres,1
1457,VAN,Vancouver Canucks,1
1453,SJS,San Jose Sharks,1
1436,CHI,Chicago Blackhawks,1
1451,PHO,Phoenix Coyotes,2
1452,PIT,Pittsburgh Penguins,2
1439,DET,Detroit Red Wings,2
1449,OTT,Ottawa Senators,2


### Rank observation

The meaning of the `rank` column changes across the dataset.

In the 1909 season, `rank` gives a complete ordering from 1 to 7.
However, in the 2009 season, several teams share the same rank.

This suggests that `rank` may represent a ranking within a division or conference
rather than a global league ranking.

In [ ]:
# Inspect rank together with conference and division in the most recent season
df[df["year"] == df["year"].max()][
    ["tmID", "name", "confID", "divID", "rank"]
].sort_values(["confID", "divID", "rank"])

,tmID,name,confID,divID,rank
1446,NJD,New Jersey Devils,EC,AT,1
1452,PIT,Pittsburgh Penguins,EC,AT,2
1450,PHI,Philadelphia Flyers,EC,AT,3
1448,NYR,New York Rangers,EC,AT,4
1447,NYI,New York Islanders,EC,AT,5
1432,BUF,Buffalo Sabres,EC,NE,1
1449,OTT,Ottawa Senators,EC,NE,2
1431,BOS,Boston Bruins,EC,NE,3
1444,MTL,Montreal Canadiens,EC,NE,4
1456,TOR,Toronto Maple Leafs,EC,NE,5


### Rank interpretation

In recent seasons, the `rank` column represents the final position within each division.

Each division has its own ranking starting from 1.
Therefore, `rank` should not be interpreted as a global league ranking.

In [ ]:
# Show teams from the most recent season ordered by points
df[df["year"] == df["year"].max()][
    ["tmID", "name", "confID", "divID", "rank", "Pts"]
].sort_values("Pts", ascending=False)

,tmID,name,confID,divID,rank,Pts
1458,WAS,Washington Capitals,EC,SE,1,121
1453,SJS,San Jose Sharks,WC,PC,1,113
1436,CHI,Chicago Blackhawks,WC,CE,1,112
1451,PHO,Phoenix Coyotes,WC,PC,2,107
1457,VAN,Vancouver Canucks,WC,NW,1,103
1446,NJD,New Jersey Devils,EC,AT,1,103
1439,DET,Detroit Red Wings,WC,CE,2,102
1452,PIT,Pittsburgh Penguins,EC,AT,2,101
1442,LAK,Los Angeles Kings,WC,PC,3,101
1445,NAS,Nashville Predators,WC,CE,3,100


### Points observation

`Pts` provides a numerical measure of team performance during the regular season.

Teams with more points generally achieve better divisional rankings.
Unlike `rank`, points can also be compared across teams from different divisions.

Therefore, `Pts` may be useful as a target for a regression-based approach.

In [ ]:
# Compare points and rank within each division in the most recent season
df[df["year"] == df["year"].max()][
    ["divID", "tmID", "name", "rank", "Pts"]
].sort_values(["divID", "rank"])

,divID,tmID,name,rank,Pts
1446,AT,NJD,New Jersey Devils,1,103
1452,AT,PIT,Pittsburgh Penguins,2,101
1450,AT,PHI,Philadelphia Flyers,3,88
1448,AT,NYR,New York Rangers,4,87
1447,AT,NYI,New York Islanders,5,79
1436,CE,CHI,Chicago Blackhawks,1,112
1439,CE,DET,Detroit Red Wings,2,102
1445,CE,NAS,Nashville Predators,3,100
1454,CE,STL,St. Louis Blues,4,90
1435,CE,CBS,Columbus Blue Jackets,5,79


### Relationship between points and rank

Within each division, teams with more points generally have a better final rank.

This confirms that `Pts` is strongly related to the divisional ranking and may be
used as a numerical target in a regression approach.

### Missing values

No missing values were found in the `Teams` dataset.

This means that, at this stage, no missing-value treatment is required for this table.

In [ ]:
# Display column data types
df.dtypes

year          int64
lgID            str
tmID            str
franchID        str
confID          str
divID           str
rank          int64
playoff         str
G             int64
W             int64
L             int64
T               str
OTL             str
Pts           int64
SoW             str
SoL             str
GF            int64
GA            int64
name            str
PIM             str
BenchMinor      str
PPG             str
PPC             str
SHA             str
PKG             str
PKC             str
SHF             str
dtype: object

### Data types

Most main performance variables are numeric, such as `rank`, `G`, `W`, `L`, `Pts`, `GF` and `GA`.

However, some columns that appear to represent numerical statistics were loaded as strings.
These columns may require further inspection and conversion during data preparation.

In [59]:
# Inspect the different values stored in the T column
df["T"].unique()

<StringArray>
['  0.0', '  1.0', '  2.0', '  4.0', '  3.0', '  5.0', '  6.0', ' 11.0',
 '  7.0', '  9.0', ' 10.0', '  8.0', ' 15.0', ' 12.0', ' 13.0', ' 16.0',
 ' 14.0', ' 19.0', ' 18.0', ' 21.0', ' 17.0', ' 22.0', ' 23.0', ' 24.0',
 ' 20.0', '     ']
Length: 26, dtype: str

### Numeric columns stored as text

Some numerical columns are stored as strings because blank values are represented
by whitespace instead of proper missing values.

For example, the `T` column contains numerical values such as `0.0`, `1.0`, etc.,
but also contains blank whitespace values.

These columns will need to be cleaned and converted to numeric types during data preparation.

In [ ]:
# Count blank values in the T column
(df["T"].str.strip() == "").sum()

np.int64(150)

### Blank values in `T`

The `T` column contains 150 blank values represented as whitespace.

These values are not detected as missing values by `isna()` and will need to be converted to proper missing values during data preparation.

In [ ]:
# Count blank whitespace values in string columns that should be numeric
numeric_like_columns = [
    "T", "OTL", "SoW", "SoL", "PIM", "BenchMinor",
    "PPG", "PPC", "SHA", "PKG", "PKC", "SHF"
]

for col in numeric_like_columns:
    print(col, (df[col].str.strip() == "").sum())

T 150
OTL 1161
SoW 1309
SoL 1309
PIM 110
BenchMinor 491
PPG 418
PPC 419
SHA 418
PKG 418
PKC 419
SHF 418


### Hidden blank values

Several columns that should be numeric contain blank values represented as whitespace.

The largest numbers of blank values appear in `OTL`, `SoW` and `SoL`.

Some of these missing values may be related to changes in hockey rules and statistics across historical seasons, so they should not be filled or removed before further analysis.

In [ ]:
# Show the number of rows and columns in the dataset
df.shape

(1459, 27)

### Dataset size

The `Teams` dataset contains 1459 rows and 27 columns.

Since several statistics are unavailable for a large portion of the historical data,
the time period used for modelling may need to be restricted later.

In [ ]:
# Find the first season where OTL has a non-blank value
df.loc[df["OTL"].str.strip() != "", "year"].min()

np.int64(1999)

### Historical availability of statistics

The `OTL` statistic only starts being available from the 1999 season.

This confirms that some attributes are not consistently available across the full historical period.
The modelling period may therefore need to be restricted later.

In [ ]:
# Find the first season where SoW has a non-blank value
df.loc[df["SoW"].str.strip() != "", "year"].min()

# Find the first season where SoL has a non-blank value
df.loc[df["SoL"].str.strip() != "", "year"].min()

np.int64(2005)

### Shootout statistics availability

The `SoW` and `SoL` statistics only start being available from the 2005 season.

These features cannot be used consistently across the full historical dataset.

In [ ]:
# Check blank values in the main performance columns
main_columns = ["G", "W", "L", "Pts", "GF", "GA"]

for col in main_columns:
    print(col, df[col].isna().sum())

G 0
W 0
L 0
Pts 0
GF 0
GA 0


### Main performance statistics

The main performance variables `G`, `W`, `L`, `Pts`, `GF` and `GA`
have no missing values.

These variables are consistently available and are good candidates for further analysis.

In [ ]:
# Inspect the number of games played across seasons
df.groupby("year")["G"].agg(["min", "max"])

,min,max
year,,
1909,12,12
1910,16,16
1911,15,18
1912,13,20
1913,16,20
...,...,...
2005,82,82
2006,82,82
2007,82,82


### Number of games across seasons

The number of games played per season changes considerably over time.

Early seasons contain far fewer games, while recent seasons use an 82-game schedule.

Because of this, raw statistics such as wins, losses, goals for and goals against
may not be directly comparable across the entire historical period.

In [ ]:
# Show recent seasons and the number of games played
df.groupby("year")["G"].agg(["min", "max"]).tail(30)

,min,max
year,,
1979,80,80
1980,80,80
1981,80,80
1982,80,80
1983,80,80
1984,80,80
1985,80,80
1986,80,80
1987,80,80


### Stable recent schedule

The number of games becomes much more stable in recent seasons.

From 1995 onwards, teams generally play 82 games per season.
This recent period may therefore be more suitable for modelling than the full historical dataset.

In [ ]:
# Count the number of seasons from 1995 onwards
df.loc[df["year"] >= 1995, "year"].nunique()

14

### Recent modelling period

From 1995 onwards, the dataset contains 14 seasons.

This period has a more stable 82-game schedule and may be a useful candidate
for the first modelling experiments.

In [ ]:
# Count the number of team-season records from 1995 onwards
df[df["year"] >= 1995].shape

(403, 27)

### Recent period size

From 1995 onwards, the dataset contains 403 team-season records.

This provides a reasonable amount of data for initial modelling experiments,
although the final training period still needs to be decided.

In [ ]:
# Display summary statistics for points from 1995 onwards
df.loc[df["year"] >= 1995, "Pts"].describe()

count    403.000000
mean      87.146402
std       15.599812
min       39.000000
25%       76.500000
50%       89.000000
75%       98.000000
max      131.000000
Name: Pts, dtype: float64

### Points distribution

From 1995 onwards, teams have an average of about 87 points per season.

The values range from 39 to 131 points, with a median of 89.
No obviously invalid values are visible from the summary statistics alone.

In [ ]:
# Create a subset with seasons from 1995 onwards
recent_teams = df[df["year"] >= 1995]

# Show the team-season with the minimum number of points
recent_teams.loc[
    recent_teams["Pts"].idxmin(),
    ["year", "tmID", "name", "Pts", "W", "L"]
]

year                                      1999
tmID                                      ATL 
name     Atlanta Thrashers                    
Pts                                         39
W                                           14
L                                           57
Name: 1162, dtype: object

In [60]:
# Create a subset with seasons from 1995 onwards
recent_teams = df[df["year"] >= 1995]

# Show the team-season with the maximum number of points
recent_teams.loc[
    recent_teams["Pts"].idxmax(),
    ["year", "tmID", "name", "Pts", "W", "L"]
]

year                                      1995
tmID                                      DET 
name     Detroit Red Wings                    
Pts                                        131
W                                           62
L                                           13
Name: 1063, dtype: object

### Extreme values in recent seasons

For seasons from 1995 onwards, the minimum number of points is 39
and the maximum is 131.

These extreme values correspond to real team-seasons and do not appear
to be data errors.

In [61]:
# Check the correlation between points and divisional rank
recent_teams[["Pts", "rank"]].corr()

,Pts,rank
Pts,1.000000,-0.822462
rank,-0.822462,1.000000


### Correlation between points and rank

There is a strong negative correlation between `Pts` and `rank`
(approximately -0.82).

This is expected because teams with more points tend to have better
(lower-numbered) divisional ranks.

In [47]:
# Check correlations between points and basic performance variables
recent_teams[["Pts", "W", "L", "GF", "GA"]].corr()

,Pts,W,L,GF,GA
Pts,1.000000,0.969739,-0.966669,0.652105,-0.676774
W,0.969739,1.000000,-0.874909,0.692207,-0.582670
L,-0.966669,-0.874909,1.000000,-0.567620,0.731425
GF,0.652105,0.692207,-0.567620,1.000000,-0.007877
GA,-0.676774,-0.582670,0.731425,-0.007877,1.000000


### Correlation with performance variables

`Pts` is very strongly correlated with wins (`W`) and losses (`L`).

It also shows a moderate relationship with goals for (`GF`) and goals against (`GA`).

These variables may be useful when creating predictive features for team performance.

## Recent Seasons Analysis (1995–2009)
From this point onwards, the analysis focuses on the more recent seasons (1995–2009),
where the number of games per season is more stable and the available statistics are more consistent.

In [48]:
# Sort the dataset by team and season
recent_teams = recent_teams.sort_values(["tmID", "year"])

In [49]:
# Create features from the previous season
recent_teams["previous_W"] = recent_teams.groupby("tmID")["W"].shift(1)
recent_teams["previous_L"] = recent_teams.groupby("tmID")["L"].shift(1)
recent_teams["previous_GF"] = recent_teams.groupby("tmID")["GF"].shift(1)
recent_teams["previous_GA"] = recent_teams.groupby("tmID")["GA"].shift(1)
recent_teams["previous_Pts"] = recent_teams.groupby("tmID")["Pts"].shift(1)

In [50]:
# Inspect the newly created previous-season features
recent_teams[
    ["year", "tmID", "Pts", "previous_Pts", "W", "previous_W"]
].head(15)

,year,tmID,Pts,previous_Pts,W,previous_W
1056,1995,ANA,78,NaN,35,NaN
1082,1996,ANA,85,78.0,36,35.0
1108,1997,ANA,65,85.0,26,36.0
1134,1998,ANA,83,65.0,35,26.0
1161,1999,ANA,83,83.0,34,35.0
1189,2000,ANA,66,83.0,25,34.0
1219,2001,ANA,69,66.0,29,25.0
1249,2002,ANA,95,69.0,40,29.0
1279,2003,ANA,76,95.0,29,40.0
1309,2005,ANA,98,76.0,43,29.0


In [51]:
# Inspect team and franchise identifiers for Anaheim
recent_teams[
    recent_teams["tmID"].isin(["ANA", "AND"])
][["year", "tmID", "franchID", "name", "Pts"]]

,year,tmID,franchID,name,Pts


### Team identifier observation

The previous-season features reveal that `tmID` can change over time.

This may break the historical continuity of a team when using `tmID` to create
lagged features. The `franchID` attribute should be inspected as a possible
more stable identifier.

In [52]:
# Inspect team IDs exactly as stored
recent_teams["tmID"].unique()[:30]

<StringArray>
[' ANA ', ' AND ', ' ATL ', ' BOS ', ' BUF ', ' CAL ', ' CAR ', ' CBS ',
 ' CHI ', ' COL ', ' DAL ', ' DET ', ' EDM ', ' FLO ', ' HAR ', ' LAK ',
 ' MIN ', ' MTL ', ' NAS ', ' NJD ', ' NYI ', ' NYR ', ' OTT ', ' PHI ',
 ' PHO ', ' PIT ', ' SJS ', ' STL ', ' TBL ', ' TOR ']
Length: 30, dtype: str

In [53]:
# Remove leading and trailing spaces from identifier columns
recent_teams["tmID"] = recent_teams["tmID"].str.strip()
recent_teams["franchID"] = recent_teams["franchID"].str.strip()

In [54]:
# Inspect team and franchise identifiers for Anaheim
recent_teams[
    recent_teams["tmID"].isin(["ANA", "AND"])
][["year", "tmID", "franchID", "name", "Pts"]]

,year,tmID,franchID,name,Pts
1056,1995,ANA,ANA,Mighty Ducks of Anaheim,78
1082,1996,ANA,ANA,Mighty Ducks of Anaheim,85
1108,1997,ANA,ANA,Mighty Ducks of Anaheim,65
1134,1998,ANA,ANA,Mighty Ducks of Anaheim,83
1161,1999,ANA,ANA,Mighty Ducks of Anaheim,83
1189,2000,ANA,ANA,Mighty Ducks of Anaheim,66
1219,2001,ANA,ANA,Mighty Ducks of Anaheim,69
1249,2002,ANA,ANA,Mighty Ducks of Anaheim,95
1279,2003,ANA,ANA,Mighty Ducks of Anaheim,76
1309,2005,ANA,ANA,Mighty Ducks of Anaheim,98


### Franchise identifier

The `tmID` of a team may change over time, while `franchID` remains stable.

For example, Anaheim changes from `ANA` to `AND`, but its `franchID` remains `ANA`.

Therefore, `franchID` will be used to link a franchise across seasons when creating previous-season features.

In [64]:
# Sort the dataset by franchise and season
recent_teams = recent_teams.sort_values(["franchID", "year"])

# Remove leading and trailing spaces from identifier columns
recent_teams["tmID"] = recent_teams["tmID"].str.strip()
recent_teams["franchID"] = recent_teams["franchID"].str.strip()

# Create features from the previous season using the franchise identifier
recent_teams["previous_W"] = recent_teams.groupby("franchID")["W"].shift(1)
recent_teams["previous_L"] = recent_teams.groupby("franchID")["L"].shift(1)
recent_teams["previous_GF"] = recent_teams.groupby("franchID")["GF"].shift(1)
recent_teams["previous_GA"] = recent_teams.groupby("franchID")["GA"].shift(1)
recent_teams["previous_Pts"] = recent_teams.groupby("franchID")["Pts"].shift(1)

In [65]:
# Verify previous-season features for Anaheim
recent_teams[
    recent_teams["franchID"] == "ANA"
][["year", "tmID", "franchID", "Pts", "previous_Pts"]]

,year,tmID,franchID,Pts,previous_Pts
1056,1995,ANA,ANA,78,NaN
1082,1996,ANA,ANA,85,78.0
1108,1997,ANA,ANA,65,85.0
1134,1998,ANA,ANA,83,65.0
1161,1999,ANA,ANA,83,83.0
1189,2000,ANA,ANA,66,83.0
1219,2001,ANA,ANA,69,66.0
1249,2002,ANA,ANA,95,69.0
1279,2003,ANA,ANA,76,95.0
1309,2005,ANA,ANA,98,76.0


### Previous-season features

Using `franchID` preserves team continuity across seasons even when `tmID` changes.

For Anaheim, the 2006 season correctly uses the 2005 points value (98) as `previous_Pts`.

In [66]:
# Count missing values in the previous-season features
recent_teams[
    ["previous_W", "previous_L", "previous_GF", "previous_GA", "previous_Pts"]
].isna().sum()

previous_W      30
previous_L      30
previous_GF     30
previous_GA     30
previous_Pts    30
dtype: int64

### Missing previous-season features

There are 30 missing values in each previous-season feature.

These correspond to franchises for which no previous season is available in the selected period.

In [67]:
# Inspect rows without previous-season information
recent_teams[
    recent_teams["previous_Pts"].isna()
][["year", "tmID", "franchID", "name", "Pts"]]

,year,tmID,franchID,name,Pts
1056,1995,ANA,ANA,Mighty Ducks of Anaheim,78
1057,1995,BOS,BOS,Boston Bruins,91
1058,1995,BUF,BUF,Buffalo Sabres,73
1059,1995,CAL,CAL,Calgary Flames,79
1066,1995,HAR,CAR,Hartford Whalers,77
1195,2000,CBS,CBS,Columbus Blue Jackets,71
1060,1995,CHI,CHI,Chicago Blackhawks,94
1061,1995,COL,COL,Colorado Avalanche,104
1062,1995,DAL,DAL,Dallas Stars,66
1063,1995,DET,DET,Detroit Red Wings,131


### Missing previous-season values

The missing values in the previous-season features occur when no earlier season is available in the selected dataset.

This happens mainly for:
- franchises whose first available season in the selected period is 1995;
- expansion/new franchises that entered the league later.

These rows cannot provide previous-season features and will need to be handled before modelling.

In [68]:
# Remove rows without previous-season information
model_data = recent_teams.dropna(
    subset=["previous_W", "previous_L", "previous_GF", "previous_GA", "previous_Pts"]
)

In [69]:
# Show the size of the modelling dataset
model_data.shape

(373, 32)

### Modelling dataset size

After removing team-seasons without previous-season information, the dataset contains
373 observations and 32 columns.

In [70]:
# Check missing values in previous-season features
model_data[
    ["previous_W", "previous_L", "previous_GF", "previous_GA", "previous_Pts"]
].isna().sum()

previous_W      0
previous_L      0
previous_GF     0
previous_GA     0
previous_Pts    0
dtype: int64

### Final check of previous-season features

The selected previous-season features contain no missing values after removing rows
without historical information.

The modelling dataset is now ready for the initial definition of predictors and target.

In [71]:
# Define predictive features
X = model_data[
    ["previous_W", "previous_L", "previous_GF", "previous_GA", "previous_Pts"]
]

# Define the target variable
y = model_data["Pts"]

In [72]:
X.head()

,previous_W,previous_L,previous_GF,previous_GA,previous_Pts
1082,35.0,39.0,234.0,247.0,78.0
1108,36.0,33.0,245.0,233.0,85.0
1134,26.0,43.0,205.0,261.0,65.0
1161,35.0,34.0,215.0,206.0,83.0
1189,34.0,33.0,217.0,227.0,83.0


In [73]:
y.head()

1082    85
1108    65
1134    83
1161    83
1189    66
Name: Pts, dtype: int64

### Predictor and target definition

The predictive variables (`X`) correspond to team performance in the previous season.

The target (`y`) is the number of points obtained in the current season.

This setup avoids using information from the season being predicted.

In [74]:
# Show the available seasons in the modelling dataset
sorted(model_data["year"].unique())

[np.int64(1996),
 np.int64(1997),
 np.int64(1998),
 np.int64(1999),
 np.int64(2000),
 np.int64(2001),
 np.int64(2002),
 np.int64(2003),
 np.int64(2005),
 np.int64(2006),
 np.int64(2007),
 np.int64(2008),
 np.int64(2009)]

## Train-Test Split

The data is split chronologically to avoid using future information during training.

- Training seasons: 1996–2006
- Test seasons: 2007–2009

In [75]:
# Split the data chronologically
train_data = model_data[model_data["year"] <= 2006]
test_data = model_data[model_data["year"] >= 2007]

In [76]:
# Show training and test dataset sizes
train_data.shape, test_data.shape

((283, 32), (90, 32))

### Train-test split result

The chronological split produced:

- 283 observations for training
- 90 observations for testing

This preserves the temporal order of the data and avoids training on future seasons.

In [77]:
# Define predictor columns
features = [
    "previous_W",
    "previous_L",
    "previous_GF",
    "previous_GA",
    "previous_Pts"
]

# Create training data
X_train = train_data[features]
y_train = train_data["Pts"]

# Create test data
X_test = test_data[features]
y_test = test_data["Pts"]

In [78]:
# Check training and test shapes
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((283, 5), (90, 5), (283,), (90,))

## First Regression Model

A simple linear regression model is used as the first baseline to predict
the points obtained by each team in the following season.

In [79]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

In [80]:
# Import evaluation metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# Calculate Mean Absolute Error
mae = mean_absolute_error(y_test, y_pred)

# Calculate Root Mean Squared Error
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

mae, rmse

(8.988317833408543, np.float64(11.308706205479595))

### Initial regression results

The first linear regression model achieved:

- MAE: 8.99 points
- RMSE: 11.31 points

This means that the model's predictions differ from the real season points by
approximately 9 points on average.

The higher RMSE indicates that some predictions have larger errors.

In [81]:
# Baseline prediction: use previous season points
baseline_pred = X_test["previous_Pts"]

# Evaluate baseline
baseline_mae = mean_absolute_error(y_test, baseline_pred)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_pred))

baseline_mae, baseline_rmse

(10.4, np.float64(12.992305415129373))

### Baseline comparison

A simple baseline was created by assuming that each team would obtain the same
number of points as in the previous season.

Baseline results:
- MAE: 10.40
- RMSE: 12.99

Linear regression results:
- MAE: 8.99
- RMSE: 11.31

The linear regression model performs better than the baseline, indicating that
the additional previous-season features provide useful predictive information.

In [82]:
# Create a comparison between real and predicted points
results = test_data[["year", "tmID", "name", "Pts"]].copy()

results["Predicted_Pts"] = y_pred
results["Error"] = results["Pts"] - results["Predicted_Pts"]

results.head(10)

,year,tmID,name,Pts,Predicted_Pts,Error
1369,2007,AND,Anaheim Ducks,102,98.858850,3.141150
1399,2008,AND,Anaheim Ducks,91,94.527090,-3.527090
1429,2009,AND,Anaheim Ducks,89,88.156001,0.843999
1371,2007,BOS,Boston Bruins,94,76.029212,17.970788
1401,2008,BOS,Boston Bruins,116,89.122101,26.877899
1431,2009,BOS,Boston Bruins,91,102.939249,-11.939249
1372,2007,BUF,Buffalo Sabres,90,98.900227,-8.900227
1402,2008,BUF,Buffalo Sabres,91,88.227671,2.772329
1432,2009,BUF,Buffalo Sabres,100,88.981340,11.018660
1373,2007,CAL,Calgary Flames,94,92.046460,1.953540


### Prediction examples

The model performs well for some team-seasons, but larger errors occur when a team's
performance changes significantly between seasons.

This suggests that previous-season team statistics alone may not be sufficient
to capture major improvements or declines.

In [83]:
# Calculate absolute error
results["Absolute_Error"] = results["Error"].abs()

# Show the 10 largest prediction errors
results.sort_values("Absolute_Error", ascending=False).head(10)

,year,tmID,name,Pts,Predicted_Pts,Error,Absolute_Error
1390,2007,PHI,Philadelphia Flyers,95,67.801096,27.198904,27.198904
1401,2008,BOS,Boston Bruins,116,89.122101,26.877899,26.877899
1451,2009,PHO,Phoenix Coyotes,107,80.395952,26.604048,26.604048
1458,2009,WAS,Washington Capitals,121,94.775257,26.224743,26.224743
1440,2009,EDM,Edmonton Oilers,62,84.499427,-22.499427,22.499427
1407,2008,COL,Colorado Avalanche,69,90.694076,-21.694076,21.694076
1417,2008,NYI,New York Islanders,61,80.611709,-19.611709,19.611709
1423,2008,SJS,San Jose Sharks,117,97.441788,19.558212,19.558212
1437,2009,COL,Colorado Avalanche,95,75.871470,19.128530,19.128530
1442,2009,LAK,Los Angeles Kings,101,82.457888,18.542112,18.542112


### Largest prediction errors

The largest errors occur in seasons where team performance changes substantially
compared with the previous season.

This suggests that previous-season team statistics alone are not enough to explain
all performance changes.

Additional information from other datasets, such as coaches, players or goalies,
may improve the predictions.

## Coaches Data Exploration

The next step is to inspect the `Coaches` dataset and identify variables
that may help explain changes in team performance between seasons.

In [85]:
# Load the Coaches dataset
coaches_df = pd.read_csv("../hockey_development_data/Coaches.csv")

# Remove leading and trailing spaces from column names
coaches_df.columns = coaches_df.columns.str.strip()

In [86]:
# Display the first rows
coaches_df.head()

,coachID,year,tmID,lgID,stint,notes,g,w,l,t,postg,postw,postl,postt
0,abelsi01c,1952,CHI,NHL,1,,70.0,27.0,28.0,15.0,7.0,3.0,4.0,0.0
1,abelsi01c,1953,CHI,NHL,1,,70.0,12.0,51.0,7.0,,,,
2,abelsi01c,1957,DET,NHL,2,,33.0,16.0,12.0,5.0,4.0,0.0,4.0,0.0
3,abelsi01c,1958,DET,NHL,1,,70.0,25.0,37.0,8.0,,,,
4,abelsi01c,1959,DET,NHL,1,,70.0,26.0,29.0,15.0,6.0,2.0,4.0,0.0


In [87]:
# Display column names
coaches_df.columns.tolist()

['coachID',
 'year',
 'tmID',
 'lgID',
 'stint',
 'notes',
 'g',
 'w',
 'l',
 't',
 'postg',
 'postw',
 'postl',
 'postt']

In [88]:
# Show dataset size
coaches_df.shape

(1741, 14)

In [89]:
# Count how many coaches each team has in each season
coaches_df.groupby(["year", "tmID"])["coachID"].nunique().value_counts().sort_index()

coachID
1    1202
2     222
3      30
Name: count, dtype: int64

### Number of coaches per team-season

Most team-seasons have a single coach.

However, some teams have two or even three coaches in the same season,
which indicates that coaching changes can occur during the season.

This must be considered before creating coach-related features.

In [90]:
# Show examples of team-seasons with more than one coach
coach_counts = coaches_df.groupby(["year", "tmID"])["coachID"].nunique()

coach_counts[coach_counts > 1].head(10)

year  tmID
1913  MOW     2
1919  TRS     2
1921  MTL     2
1922  TRS     2
1923  HAM     2
1926  DTC     2
      TRS     3
1927  CHI     2
1928  CHI     2
1929  CHI     2
Name: coachID, dtype: int64

In [91]:
# Inspect a team-season with multiple coaches
coaches_df[
    (coaches_df["year"] == 1913) &
    (coaches_df["tmID"].str.strip() == "MOW")
]

,coachID,year,tmID,lgID,stint,notes,g,w,l,t,postg,postw,postl,postt
144,boondi01c,1913,MOW,NHA,1,,15.0,4.0,11.0,0.0,,,,
1379,rossar01c,1913,MOW,NHA,2,,5.0,3.0,2.0,0.0,,,,


### Coach stint

The `stint` column indicates the order in which coaches managed a team during a season.

For example, in 1913, team `MOW` had two coaches:
- `stint = 1`: first coach
- `stint = 2`: second coach

This confirms that coaching changes can occur within the same season.

In [92]:
# Restrict coaches data to the recent modelling period
recent_coaches = coaches_df[coaches_df["year"] >= 1995]

# Count coaches per team-season in the recent period
recent_coach_counts = (
    recent_coaches
    .groupby(["year", "tmID"])["coachID"]
    .nunique()
)

recent_coach_counts.value_counts().sort_index()

coachID
1    327
2     66
3     10
Name: count, dtype: int64

### Coaching changes in recent seasons

In the 1995–2009 period:

- 327 team-seasons had one coach
- 66 team-seasons had two coaches
- 10 team-seasons had three coaches

This shows that in-season coaching changes occur frequently enough to be considered in the analysis.

In [93]:
# Create a binary indicator for in-season coaching changes
coach_change_df = recent_coach_counts.reset_index(name="num_coaches")

coach_change_df["coach_change"] = (
    coach_change_df["num_coaches"] > 1
).astype(int)

coach_change_df.head()

,year,tmID,num_coaches,coach_change
0,1995,ANA,1,0
1,1995,BOS,1,0
2,1995,BUF,1,0
3,1995,CAL,1,0
4,1995,CHI,1,0


In [94]:
# Remove leading and trailing spaces from team identifiers
recent_teams["tmID"] = recent_teams["tmID"].str.strip()
coach_change_df["tmID"] = coach_change_df["tmID"].str.strip()

In [95]:
# Merge coaching-change information with the team dataset
recent_teams = recent_teams.merge(
    coach_change_df,
    on=["year", "tmID"],
    how="left"
)

In [96]:
# Inspect the new coaching-related columns
recent_teams[
    ["year", "tmID", "name", "num_coaches", "coach_change"]
].head(10)

,year,tmID,name,num_coaches,coach_change
0,1995,ANA,Mighty Ducks of Anaheim,1,0
1,1996,ANA,Mighty Ducks of Anaheim,1,0
2,1997,ANA,Mighty Ducks of Anaheim,1,0
3,1998,ANA,Mighty Ducks of Anaheim,1,0
4,1999,ANA,Mighty Ducks of Anaheim,1,0
5,2000,ANA,Mighty Ducks of Anaheim,2,1
6,2001,ANA,Mighty Ducks of Anaheim,1,0
7,2002,ANA,Mighty Ducks of Anaheim,1,0
8,2003,ANA,Mighty Ducks of Anaheim,1,0
9,2005,ANA,Mighty Ducks of Anaheim,1,0


In [97]:
# Sort by franchise and season
recent_teams = recent_teams.sort_values(["franchID", "year"])

# Create previous-season coaching features
recent_teams["previous_num_coaches"] = (
    recent_teams.groupby("franchID")["num_coaches"].shift(1)
)

recent_teams["previous_coach_change"] = (
    recent_teams.groupby("franchID")["coach_change"].shift(1)
)

In [98]:
# Inspect previous coaching features for Anaheim
recent_teams[
    recent_teams["franchID"] == "ANA"
][[
    "year",
    "tmID",
    "num_coaches",
    "coach_change",
    "previous_num_coaches",
    "previous_coach_change"
]]

,year,tmID,num_coaches,coach_change,previous_num_coaches,previous_coach_change
0,1995,ANA,1,0,NaN,NaN
1,1996,ANA,1,0,1.0,0.0
2,1997,ANA,1,0,1.0,0.0
3,1998,ANA,1,0,1.0,0.0
4,1999,ANA,1,0,1.0,0.0
5,2000,ANA,2,1,1.0,0.0
6,2001,ANA,1,0,2.0,1.0
7,2002,ANA,1,0,1.0,0.0
8,2003,ANA,1,0,1.0,0.0
9,2005,ANA,1,0,1.0,0.0


### Previous-season coaching features

The previous-season coaching features were created successfully.

For example, Anaheim had a coaching change in 2000, and this information is correctly
represented as `previous_coach_change = 1` for the 2001 season.

These features can now be tested as additional predictors of team performance.

In [99]:
# Remove rows without previous-season information
model_data = recent_teams.dropna(
    subset=[
        "previous_W",
        "previous_L",
        "previous_GF",
        "previous_GA",
        "previous_Pts",
        "previous_num_coaches",
        "previous_coach_change"
    ]
)

model_data.shape

(373, 36)

## Model with Coaching Features

The previous model only used team performance from the previous season.

A second model is now created including coaching-related information from the
previous season, in order to test whether coaching changes improve prediction accuracy.

In [104]:
# Split the updated modelling dataset chronologically
train_data = model_data[model_data["year"] <= 2006]
test_data = model_data[model_data["year"] >= 2007]

In [105]:
# Define features including coaching information
features_with_coaches = [
    "previous_W",
    "previous_L",
    "previous_GF",
    "previous_GA",
    "previous_Pts",
    "previous_num_coaches",
    "previous_coach_change"
]

In [106]:
# Create training and test data
X_train = train_data[features_with_coaches]
y_train = train_data["Pts"]

X_test = test_data[features_with_coaches]
y_test = test_data["Pts"]

In [107]:
X_train.shape, X_test.shape

((283, 7), (90, 7))

In [108]:
# Train the linear regression model with coaching features
model_with_coaches = LinearRegression()

model_with_coaches.fit(X_train, y_train)

# Make predictions
y_pred_coaches = model_with_coaches.predict(X_test)

In [109]:
# Evaluate the model
mae_coaches = mean_absolute_error(y_test, y_pred_coaches)
rmse_coaches = np.sqrt(mean_squared_error(y_test, y_pred_coaches))

mae_coaches, rmse_coaches

(9.187600097986088, np.float64(11.442609957944951))

### Model Comparison

The model including previous-season coaching features achieved:

- MAE: 9.19
- RMSE: 11.44

The previous model, using only team performance features, achieved:

- MAE: 8.99
- RMSE: 11.31

Therefore, adding `previous_num_coaches` and `previous_coach_change` did not improve
the predictive performance of the linear regression model.

This suggests that these simple coaching features do not provide additional useful
information for predicting next-season points.

## Scoring Data Exploration

The `Scoring` dataset contains player-level offensive statistics.

The goal of this exploration is to identify variables that can be aggregated at the
team level and used as previous-season predictors of team performance.

In [2]:
import pandas as pd

# Load the Scoring dataset
scoring_df = pd.read_csv("../hockey_development_data/Scoring.csv")

# Remove leading and trailing spaces from column names
scoring_df.columns = scoring_df.columns.str.strip()

In [3]:
scoring_df.head()

,playerID,year,stint,tmID,lgID,pos,GP,G,A,Pts,...,PostA,PostPts,PostPIM,Post+/-,PostPPG,PostPPA,PostSHG,PostSHA,PostGWG,PostSOG
0,aaltoan01,1997,1,ANA,NHL,C,3.0,0.0,0.0,0.0,...,,,,,,,,,,
1,aaltoan01,1998,1,ANA,NHL,C,73.0,3.0,5.0,8.0,...,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,aaltoan01,1999,1,ANA,NHL,C,63.0,7.0,11.0,18.0,...,,,,,,,,,,
3,aaltoan01,2000,1,ANA,NHL,C,12.0,1.0,1.0,2.0,...,,,,,,,,,,
4,abbeybr01,1975,1,CIN,WHA,D,17.0,1.0,0.0,1.0,...,,,,,,,,,,


In [4]:
scoring_df.columns

Index(['playerID', 'year', 'stint', 'tmID', 'lgID', 'pos', 'GP', 'G', 'A',
       'Pts', 'PIM', '+/-', 'PPG', 'PPA', 'SHG', 'SHA', 'GWG', 'GTG', 'SOG',
       'PostGP', 'PostG', 'PostA', 'PostPts', 'PostPIM', 'Post+/-', 'PostPPG',
       'PostPPA', 'PostSHG', 'PostSHA', 'PostGWG', 'PostSOG'],
      dtype='str')

In [5]:
scoring_df.shape

(43848, 31)

In [6]:
# Restrict to the recent modelling period
recent_scoring = scoring_df[scoring_df["year"] >= 1995].copy()

# Remove leading and trailing spaces from team identifiers
recent_scoring["tmID"] = recent_scoring["tmID"].str.strip()

In [7]:
recent_scoring.shape

(14537, 31)

In [8]:
recent_scoring[["GP", "G", "A", "Pts", "SOG"]].dtypes

GP     str
G      str
A      str
Pts    str
SOG    str
dtype: object

### Scoring Data Types

The main player performance variables are stored as text rather than numeric values.

Before aggregating player statistics at the team level, these columns must be converted to numeric format.

In [9]:
# Convert selected scoring columns to numeric
scoring_numeric_cols = ["GP", "G", "A", "Pts", "SOG"]

for col in scoring_numeric_cols:
    recent_scoring[col] = pd.to_numeric(
        recent_scoring[col].str.strip(),
        errors="coerce"
    )

In [10]:
recent_scoring[scoring_numeric_cols].dtypes

GP     float64
G      float64
A      float64
Pts    float64
SOG    float64
dtype: object

In [11]:
# Aggregate player statistics at the team-season level
team_scoring = (
    recent_scoring
    .groupby(["year", "tmID"])[["G", "A", "Pts", "SOG"]]
    .sum()
    .reset_index()
)

team_scoring.head()

,year,tmID,G,A,Pts,SOG
0,1995,ANA,234.0,403.0,637.0,2354.0
1,1995,BOS,282.0,457.0,739.0,2838.0
2,1995,BUF,247.0,417.0,664.0,2333.0
3,1995,CAL,241.0,383.0,624.0,2466.0
4,1995,CHI,273.0,475.0,748.0,2240.0


In [12]:
# Rename aggregated scoring columns
team_scoring = team_scoring.rename(columns={
    "G": "player_G",
    "A": "player_A",
    "Pts": "player_Pts",
    "SOG": "player_SOG"
})

team_scoring.head()

,year,tmID,player_G,player_A,player_Pts,player_SOG
0,1995,ANA,234.0,403.0,637.0,2354.0
1,1995,BOS,282.0,457.0,739.0,2838.0
2,1995,BUF,247.0,417.0,664.0,2333.0
3,1995,CAL,241.0,383.0,624.0,2466.0
4,1995,CHI,273.0,475.0,748.0,2240.0


In [14]:
# Load the Teams dataset
df = pd.read_csv("../hockey_development_data/Teams.csv")

# Clean column names
df.columns = df.columns.str.strip()

# Clean team and franchise identifiers
df["tmID"] = df["tmID"].str.strip()
df["franchID"] = df["franchID"].str.strip()

# Restrict to the recent modelling period
recent_teams = df[df["year"] >= 1995].copy()

In [15]:
# Merge aggregated scoring data with team data
recent_teams = recent_teams.merge(
    team_scoring,
    on=["year", "tmID"],
    how="left"
)

In [16]:
recent_teams[
    ["year", "tmID", "name", "player_G", "player_A", "player_Pts", "player_SOG"]
].head(10)

,year,tmID,name,player_G,player_A,player_Pts,player_SOG
0,1995,ANA,Mighty Ducks of Anaheim,234.0,403.0,637.0,2354.0
1,1995,BOS,Boston Bruins,282.0,457.0,739.0,2838.0
2,1995,BUF,Buffalo Sabres,247.0,417.0,664.0,2333.0
3,1995,CAL,Calgary Flames,241.0,383.0,624.0,2466.0
4,1995,CHI,Chicago Blackhawks,273.0,475.0,748.0,2240.0
5,1995,COL,Colorado Avalanche,326.0,542.0,868.0,2644.0
6,1995,DAL,Dallas Stars,227.0,377.0,604.0,2472.0
7,1995,DET,Detroit Red Wings,325.0,555.0,880.0,2648.0
8,1995,EDM,Edmonton Oilers,240.0,405.0,645.0,2459.0
9,1995,FLO,Florida Panthers,254.0,434.0,688.0,2364.0


In [17]:
# Check missing values in aggregated scoring features
recent_teams[
    ["player_G", "player_A", "player_Pts", "player_SOG"]
].isna().sum()

player_G      0
player_A      0
player_Pts    0
player_SOG    0
dtype: int64

In [18]:
# Sort by franchise and season
recent_teams = recent_teams.sort_values(["franchID", "year"])

# Create previous-season scoring features
recent_teams["previous_player_G"] = (
    recent_teams.groupby("franchID")["player_G"].shift(1)
)

recent_teams["previous_player_A"] = (
    recent_teams.groupby("franchID")["player_A"].shift(1)
)

recent_teams["previous_player_Pts"] = (
    recent_teams.groupby("franchID")["player_Pts"].shift(1)
)

recent_teams["previous_player_SOG"] = (
    recent_teams.groupby("franchID")["player_SOG"].shift(1)
)

In [19]:
recent_teams[
    recent_teams["franchID"] == "ANA"
][[
    "year",
    "tmID",
    "player_Pts",
    "previous_player_Pts"
]]

,year,tmID,player_Pts,previous_player_Pts
0,1995,ANA,637.0,NaN
26,1996,ANA,662.0,637.0
52,1997,ANA,523.0,662.0
78,1998,ANA,565.0,523.0
105,1999,ANA,592.0,565.0
133,2000,ANA,511.0,592.0
163,2001,ANA,470.0,511.0
193,2002,ANA,558.0,470.0
223,2003,ANA,498.0,558.0
253,2005,ANA,683.0,498.0


### Previous-Season Scoring Features

The aggregated player statistics were shifted by franchise to represent the previous season.

This ensures that the model only uses information available before the season being predicted.

In [20]:
# Create previous-season team performance features
recent_teams["previous_W"] = (
    recent_teams.groupby("franchID")["W"].shift(1)
)

recent_teams["previous_L"] = (
    recent_teams.groupby("franchID")["L"].shift(1)
)

recent_teams["previous_GF"] = (
    recent_teams.groupby("franchID")["GF"].shift(1)
)

recent_teams["previous_GA"] = (
    recent_teams.groupby("franchID")["GA"].shift(1)
)

recent_teams["previous_Pts"] = (
    recent_teams.groupby("franchID")["Pts"].shift(1)
)

In [21]:
model_data = recent_teams.dropna(
    subset=[
        "previous_W",
        "previous_L",
        "previous_GF",
        "previous_GA",
        "previous_Pts",
        "previous_player_G",
        "previous_player_A",
        "previous_player_Pts",
        "previous_player_SOG"
    ]
)

model_data.shape

(373, 40)

## Model with Player Scoring Features

A new linear regression model is created using both previous-season team performance
and aggregated player scoring statistics.

The goal is to test whether player-level offensive information improves the prediction
of next-season team points.

In [22]:
features_with_scoring = [
    "previous_W",
    "previous_L",
    "previous_GF",
    "previous_GA",
    "previous_Pts",
    "previous_player_G",
    "previous_player_A",
    "previous_player_Pts",
    "previous_player_SOG"
]

In [23]:
# Split the data chronologically
train_data = model_data[model_data["year"] <= 2006]
test_data = model_data[model_data["year"] >= 2007]

In [24]:
# Create training and test data
X_train = train_data[features_with_scoring]
y_train = train_data["Pts"]

X_test = test_data[features_with_scoring]
y_test = test_data["Pts"]

In [25]:
from sklearn.linear_model import LinearRegression

# Train the model
model_with_scoring = LinearRegression()
model_with_scoring.fit(X_train, y_train)

# Make predictions
y_pred_scoring = model_with_scoring.predict(X_test)

In [26]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

mae_scoring = mean_absolute_error(y_test, y_pred_scoring)
rmse_scoring = np.sqrt(mean_squared_error(y_test, y_pred_scoring))

mae_scoring, rmse_scoring

(8.96217287081292, np.float64(11.209289690754577))

### Model Comparison with Scoring Features

The model including previous-season player scoring features achieved:

- MAE: 8.96
- RMSE: 11.21

The previous model using only team performance features achieved:

- MAE: 8.99
- RMSE: 11.31

Therefore, adding aggregated player scoring information slightly improved predictive performance.

This suggests that previous-season player offensive production contains some additional information that is useful for predicting next-season team points.

## Goalies Data Exploration

The `Goalies` dataset contains goalkeeper-level performance statistics.

The goal is to identify statistics that can be aggregated at the team level and used as previous-season predictors of team performance.

In [27]:
# Load the Goalies dataset
goalies_df = pd.read_csv("../hockey_development_data/Goalies.csv")

# Remove leading and trailing spaces from column names
goalies_df.columns = goalies_df.columns.str.strip()

In [28]:
goalies_df.head()

,playerID,year,stint,tmID,lgID,GP,Min,W,L,T/OL,...,SA,PostGP,PostMin,PostW,PostL,PostT,PostENG,PostSHO,PostGA,PostSA
0,abbotge01,1943,1,BOS,NHL,1.0,60.0,0.0,1.0,0.0,...,,,,,,,,,,
1,abrahch01,1974,1,NEW,WHA,16.0,870.0,8.0,6.0,1.0,...,504.0,,,,,,,,,
2,abrahch01,1975,1,NEW,WHA,41.0,2385.0,18.0,18.0,2.0,...,1221.0,1.0,1.0,0.0,0.0,,0.0,0.0,0.0,
3,abrahch01,1976,1,NEW,WHA,45.0,2484.0,15.0,22.0,4.0,...,1438.0,2.0,90.0,0.0,1.0,,0.0,0.0,5.0,51.0
4,adamsjo02,1972,1,BOS,NHL,14.0,780.0,9.0,3.0,1.0,...,,,,,,,,,,


In [29]:
goalies_df.columns

Index(['playerID', 'year', 'stint', 'tmID', 'lgID', 'GP', 'Min', 'W', 'L',
       'T/OL', 'ENG', 'SHO', 'GA', 'SA', 'PostGP', 'PostMin', 'PostW', 'PostL',
       'PostT', 'PostENG', 'PostSHO', 'PostGA', 'PostSA'],
      dtype='str')

In [30]:
goalies_df.shape

(4096, 23)

In [31]:
# Restrict to the recent modelling period
recent_goalies = goalies_df[goalies_df["year"] >= 1995].copy()

# Clean team identifiers
recent_goalies["tmID"] = recent_goalies["tmID"].str.strip()

In [32]:
recent_goalies.shape

(1286, 23)

In [33]:
recent_goalies[
    ["GP", "Min", "W", "L", "SHO", "GA", "SA"]
].dtypes

GP     str
Min    str
W      str
L      str
SHO    str
GA     str
SA     str
dtype: object

### Goalie Data Types

The main goalie performance variables are stored as text.

These columns must be converted to numeric values before they can be aggregated at the team-season level.

In [34]:
# Convert selected goalie columns to numeric
goalie_numeric_cols = ["GP", "Min", "W", "L", "SHO", "GA", "SA"]

for col in goalie_numeric_cols:
    recent_goalies[col] = pd.to_numeric(
        recent_goalies[col].str.strip(),
        errors="coerce"
    )

In [35]:
recent_goalies[goalie_numeric_cols].dtypes

GP     float64
Min    float64
W      float64
L      float64
SHO    float64
GA     float64
SA     float64
dtype: object

In [36]:
# Aggregate goalie statistics at the team-season level
team_goalies = (
    recent_goalies
    .groupby(["year", "tmID"])[["W", "L", "SHO", "GA", "SA"]]
    .sum()
    .reset_index()
)

team_goalies.head()

,year,tmID,W,L,SHO,GA,SA
0,1995,ANA,35.0,39.0,4.0,242.0,2634.0
1,1995,BOS,40.0,31.0,2.0,266.0,2201.0
2,1995,BUF,33.0,42.0,2.0,254.0,2904.0
3,1995,CAL,34.0,37.0,6.0,236.0,2217.0
4,1995,CHI,40.0,28.0,5.0,215.0,2329.0


In [37]:
# Rename aggregated goalie columns
team_goalies = team_goalies.rename(columns={
    "W": "goalie_W",
    "L": "goalie_L",
    "SHO": "goalie_SHO",
    "GA": "goalie_GA",
    "SA": "goalie_SA"
})

team_goalies.head()

,year,tmID,goalie_W,goalie_L,goalie_SHO,goalie_GA,goalie_SA
0,1995,ANA,35.0,39.0,4.0,242.0,2634.0
1,1995,BOS,40.0,31.0,2.0,266.0,2201.0
2,1995,BUF,33.0,42.0,2.0,254.0,2904.0
3,1995,CAL,34.0,37.0,6.0,236.0,2217.0
4,1995,CHI,40.0,28.0,5.0,215.0,2329.0


In [38]:
# Merge aggregated goalie data with team data
recent_teams = recent_teams.merge(
    team_goalies,
    on=["year", "tmID"],
    how="left"
)

In [39]:
recent_teams[
    ["year", "tmID", "name", "goalie_W", "goalie_L", "goalie_SHO", "goalie_GA", "goalie_SA"]
].head(10)

,year,tmID,name,goalie_W,goalie_L,goalie_SHO,goalie_GA,goalie_SA
0,1995,ANA,Mighty Ducks of Anaheim,35.0,39.0,4.0,242.0,2634.0
1,1996,ANA,Mighty Ducks of Anaheim,36.0,33.0,6.0,227.0,2682.0
2,1997,ANA,Mighty Ducks of Anaheim,26.0,43.0,4.0,252.0,2483.0
3,1998,ANA,Mighty Ducks of Anaheim,35.0,34.0,7.0,202.0,2592.0
4,1999,ANA,Mighty Ducks of Anaheim,34.0,36.0,5.0,218.0,2250.0
5,2000,ANA,Mighty Ducks of Anaheim,25.0,46.0,6.0,240.0,2412.0
6,2001,ANA,Mighty Ducks of Anaheim,29.0,45.0,4.0,191.0,2246.0
7,2002,ANA,Mighty Ducks of Anaheim,40.0,33.0,9.0,184.0,2368.0
8,2003,ANA,Mighty Ducks of Anaheim,29.0,43.0,5.0,206.0,2436.0
9,2005,ANA,Mighty Ducks of Anaheim,43.0,27.0,3.0,216.0,2425.0


In [40]:
# Check missing values in goalie features
recent_teams[
    ["goalie_W", "goalie_L", "goalie_SHO", "goalie_GA", "goalie_SA"]
].isna().sum()

goalie_W      0
goalie_L      0
goalie_SHO    0
goalie_GA     0
goalie_SA     0
dtype: int64

In [41]:
# Sort by franchise and season
recent_teams = recent_teams.sort_values(["franchID", "year"])

# Create previous-season goalie features
recent_teams["previous_goalie_W"] = (
    recent_teams.groupby("franchID")["goalie_W"].shift(1)
)

recent_teams["previous_goalie_L"] = (
    recent_teams.groupby("franchID")["goalie_L"].shift(1)
)

recent_teams["previous_goalie_SHO"] = (
    recent_teams.groupby("franchID")["goalie_SHO"].shift(1)
)

recent_teams["previous_goalie_GA"] = (
    recent_teams.groupby("franchID")["goalie_GA"].shift(1)
)

recent_teams["previous_goalie_SA"] = (
    recent_teams.groupby("franchID")["goalie_SA"].shift(1)
)

In [42]:
recent_teams[
    recent_teams["franchID"] == "ANA"
][[
    "year",
    "tmID",
    "goalie_GA",
    "previous_goalie_GA"
]]

,year,tmID,goalie_GA,previous_goalie_GA
0,1995,ANA,242.0,NaN
1,1996,ANA,227.0,242.0
2,1997,ANA,252.0,227.0
3,1998,ANA,202.0,252.0
4,1999,ANA,218.0,202.0
5,2000,ANA,240.0,218.0
6,2001,ANA,191.0,240.0
7,2002,ANA,184.0,191.0
8,2003,ANA,206.0,184.0
9,2005,ANA,216.0,206.0


### Previous-Season Goalie Features

Goalie statistics were shifted by franchise to represent the previous season.

This ensures that the model only uses goalie information that would have been available before the season being predicted.

In [43]:
model_data = recent_teams.dropna(
    subset=[
        "previous_W",
        "previous_L",
        "previous_GF",
        "previous_GA",
        "previous_Pts",
        "previous_player_G",
        "previous_player_A",
        "previous_player_Pts",
        "previous_player_SOG",
        "previous_goalie_W",
        "previous_goalie_L",
        "previous_goalie_SHO",
        "previous_goalie_GA",
        "previous_goalie_SA"
    ]
)

model_data.shape

(373, 50)

## Model with Player Scoring and Goalie Features

A new linear regression model is created using previous-season team performance,
player scoring statistics, and goalie statistics.

The goal is to test whether goalie information provides additional predictive value
for next-season team points.

In [44]:
features_with_goalies = [
    "previous_W",
    "previous_L",
    "previous_GF",
    "previous_GA",
    "previous_Pts",
    "previous_player_G",
    "previous_player_A",
    "previous_player_Pts",
    "previous_player_SOG",
    "previous_goalie_W",
    "previous_goalie_L",
    "previous_goalie_SHO",
    "previous_goalie_GA",
    "previous_goalie_SA"
]

In [45]:
# Split the data chronologically
train_data = model_data[model_data["year"] <= 2006]
test_data = model_data[model_data["year"] >= 2007]

In [46]:
# Create training and test data
X_train = train_data[features_with_goalies]
y_train = train_data["Pts"]

X_test = test_data[features_with_goalies]
y_test = test_data["Pts"]

In [47]:
# Train the model
model_with_goalies = LinearRegression()
model_with_goalies.fit(X_train, y_train)

# Make predictions
y_pred_goalies = model_with_goalies.predict(X_test)

In [48]:
# Evaluate the model
mae_goalies = mean_absolute_error(y_test, y_pred_goalies)
rmse_goalies = np.sqrt(mean_squared_error(y_test, y_pred_goalies))

mae_goalies, rmse_goalies

(8.80756921422008, np.float64(11.16023404057189))

### Model Comparison with Goalie Features

The model including previous-season team performance, player scoring, and goalie statistics achieved:

- MAE: 8.81
- RMSE: 11.16

This is an improvement over the previous models.

The results suggest that goalie performance adds useful information for predicting next-season team points.

In [49]:
# Create a results table for the best model
results_goalies = test_data[
    ["year", "tmID", "name", "Pts"]
].copy()

results_goalies["Predicted_Pts"] = y_pred_goalies
results_goalies["Error"] = (
    results_goalies["Pts"] - results_goalies["Predicted_Pts"]
)
results_goalies["Absolute_Error"] = (
    results_goalies["Error"].abs()
)

results_goalies.head()

,year,tmID,name,Pts,Predicted_Pts,Error,Absolute_Error
11,2007,AND,Anaheim Ducks,102,103.124248,-1.124248,1.124248
12,2008,AND,Anaheim Ducks,91,98.097939,-7.097939,7.097939
13,2009,AND,Anaheim Ducks,89,94.251908,-5.251908,5.251908
25,2007,BOS,Boston Bruins,94,79.072059,14.927941,14.927941
26,2008,BOS,Boston Bruins,116,89.113657,26.886343,26.886343


In [50]:
# Show the largest prediction errors
results_goalies.sort_values(
    "Absolute_Error",
    ascending=False
).head(10)

,year,tmID,name,Pts,Predicted_Pts,Error,Absolute_Error
105,2008,COL,Colorado Avalanche,69,99.266015,-30.266015,30.266015
294,2009,PHO,Phoenix Coyotes,107,79.610732,27.389268,27.389268
26,2008,BOS,Boston Bruins,116,89.113657,26.886343,26.886343
348,2007,TBL,Tampa Bay Lightning,71,95.367569,-24.367569,24.367569
278,2007,PHI,Philadelphia Flyers,95,70.634085,24.365915,24.365915
148,2009,EDM,Edmonton Oilers,62,86.204928,-24.204928,24.204928
392,2009,WAS,Washington Capitals,121,96.955917,24.044083,24.044083
237,2008,NYI,New York Islanders,61,83.600841,-22.600841,22.600841
390,2007,WAS,Washington Capitals,94,74.257259,19.742741,19.742741
197,2007,MTL,Montreal Canadiens,104,85.700364,18.299636,18.299636


### Largest Prediction Errors

The largest prediction errors occur mainly when a team's performance changes substantially from one season to the next.

The model tends to predict continuity based on previous-season performance, making it more difficult to anticipate sudden improvements or declines.

This suggests that additional variables may be needed to capture major changes between seasons.

In [51]:
# Calculate the change in team points from the previous season
results_goalies["previous_Pts"] = test_data["previous_Pts"]

results_goalies["Pts_Change"] = (
    results_goalies["Pts"] - results_goalies["previous_Pts"]
)

results_goalies["Absolute_Pts_Change"] = (
    results_goalies["Pts_Change"].abs()
)

In [52]:
results_goalies.sort_values(
    "Absolute_Error",
    ascending=False
)[[
    "year",
    "name",
    "previous_Pts",
    "Pts",
    "Pts_Change",
    "Absolute_Error"
]].head(10)

,year,name,previous_Pts,Pts,Pts_Change,Absolute_Error
105,2008,Colorado Avalanche,95.0,69,-26.0,30.266015
294,2009,Phoenix Coyotes,79.0,107,28.0,27.389268
26,2008,Boston Bruins,94.0,116,22.0,26.886343
348,2007,Tampa Bay Lightning,93.0,71,-22.0,24.367569
278,2007,Philadelphia Flyers,56.0,95,39.0,24.365915
148,2009,Edmonton Oilers,85.0,62,-23.0,24.204928
392,2009,Washington Capitals,108.0,121,13.0,24.044083
237,2008,New York Islanders,79.0,61,-18.0,22.600841
390,2007,Washington Capitals,70.0,94,24.0,19.742741
197,2007,Montreal Canadiens,90.0,104,14.0,18.299636


### Performance Changes and Prediction Errors

The largest prediction errors are generally associated with teams whose points changed substantially from the previous season.

For example, Philadelphia improved by 39 points, while Colorado declined by 26 points.

This confirms that the current model is better at predicting stable team performance than sudden improvements or declines.

In [53]:
results_goalies[
    ["Absolute_Pts_Change", "Absolute_Error"]
].corr()

,Absolute_Pts_Change,Absolute_Error
Absolute_Pts_Change,1.000000,0.645349
Absolute_Error,0.645349,1.000000


### Prediction Error and Performance Change

The correlation between the absolute change in team points and the absolute prediction error is approximately **0.65**.

This indicates that larger changes in team performance between seasons are generally associated with larger prediction errors.

Therefore, the current model performs better when team performance remains relatively stable and has more difficulty predicting sudden improvements or declines.

In [54]:
# Create predicted ranking for each season
results_goalies["Predicted_Rank"] = (
    results_goalies
    .groupby("year")["Predicted_Pts"]
    .rank(method="min", ascending=False)
    .astype(int)
)

In [55]:
# Create actual ranking based on real points
results_goalies["Actual_Rank"] = (
    results_goalies
    .groupby("year")["Pts"]
    .rank(method="min", ascending=False)
    .astype(int)
)

In [56]:
results_goalies[
    results_goalies["year"] == 2009
][[
    "tmID",
    "name",
    "Pts",
    "Predicted_Pts",
    "Actual_Rank",
    "Predicted_Rank"
]].sort_values("Predicted_Rank").head(10)

,tmID,name,Pts,Predicted_Pts,Actual_Rank,Predicted_Rank
322,SJS,San Jose Sharks,113,105.893324,2,1
134,DET,Detroit Red Wings,102,105.322868,7,2
224,NJD,New Jersey Devils,103,102.563961,5,3
92,CHI,Chicago Blackhawks,112,102.210383,3,4
27,BOS,Boston Bruins,91,101.000256,14,5
252,NYR,New York Rangers,87,98.792859,21,6
392,WAS,Washington Capitals,121,96.955917,1,7
69,CAR,Carolina Hurricanes,80,96.523339,24,8
41,BUF,Buffalo Sabres,100,94.493206,10,9
308,PIT,Pittsburgh Penguins,101,94.302516,8,10


In [57]:
# Calculate ranking error
results_goalies["Rank_Error"] = (
    results_goalies["Actual_Rank"] - results_goalies["Predicted_Rank"]
)

results_goalies["Absolute_Rank_Error"] = (
    results_goalies["Rank_Error"].abs()
)

In [58]:
# Average ranking error
results_goalies["Absolute_Rank_Error"].mean()

np.float64(6.7555555555555555)

### Ranking Prediction Error

The average absolute ranking error is approximately **6.76 positions**.

This means that, on average, the predicted team position differs from the actual final position by about seven places.

Although the model captures part of the overall ordering, there is still considerable room for improvement, especially for teams with large year-to-year performance changes.

In [59]:
results_goalies[
    ["Actual_Rank", "Predicted_Rank"]
].corr()

,Actual_Rank,Predicted_Rank
Actual_Rank,1.000000,0.487525
Predicted_Rank,0.487525,1.000000


### Ranking Correlation

The correlation between the actual and predicted rankings is approximately **0.49**.

This indicates a moderate positive relationship between the predicted and actual team ordering.

The model captures part of the ranking structure, but its predictions are still far from perfectly matching the final league order.

## KNN Regression Model

A K-Nearest Neighbors regression model is tested using the same training and test sets as the linear regression model.

This allows a fair comparison between different regression approaches.

In [60]:
from sklearn.neighbors import KNeighborsRegressor

# Create a KNN regression model
knn_model = KNeighborsRegressor(n_neighbors=5)

# Train the model
knn_model.fit(X_train, y_train)

# Make predictions
y_pred_knn = knn_model.predict(X_test)

In [61]:
# Evaluate the KNN model
mae_knn = mean_absolute_error(y_test, y_pred_knn)
rmse_knn = np.sqrt(mean_squared_error(y_test, y_pred_knn))

mae_knn, rmse_knn

(10.697777777777777, np.float64(13.113250296296998))

### KNN Model Comparison

The KNN regression model achieved:

- MAE: 10.70
- RMSE: 13.11

These results are worse than the linear regression model, which achieved:

- MAE: 8.81
- RMSE: 11.16

Therefore, the KNN model with `k = 5` does not improve predictive performance.

## Decision Tree Regression Model

A Decision Tree regression model is tested using the same training and test sets.

This model can capture non-linear relationships between previous-season statistics and next-season team points.

In [62]:
from sklearn.tree import DecisionTreeRegressor

# Create the Decision Tree regression model
tree_model = DecisionTreeRegressor(
    max_depth=5,
    random_state=42
)

# Train the model
tree_model.fit(X_train, y_train)

# Make predictions
y_pred_tree = tree_model.predict(X_test)

In [63]:
# Evaluate the Decision Tree model
mae_tree = mean_absolute_error(y_test, y_pred_tree)
rmse_tree = np.sqrt(mean_squared_error(y_test, y_pred_tree))

mae_tree, rmse_tree

(12.3995346022417, np.float64(15.561683627528035))

### Decision Tree Model Comparison

The Decision Tree regression model achieved:

- MAE: 12.40
- RMSE: 15.56

These results are worse than both the Linear Regression and KNN models.

Therefore, with the current configuration, the Decision Tree does not improve predictive performance.

## KNN with Feature Scaling

KNN is sensitive to the scale of the input variables because it is based on distances between observations.

The features are standardized before training the model so that variables with larger numerical ranges do not dominate the distance calculation.

In [64]:
from sklearn.preprocessing import StandardScaler

# Create the scaler
scaler = StandardScaler()

# Fit on training data and transform both sets
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [65]:
# Train KNN using scaled features
knn_scaled_model = KNeighborsRegressor(n_neighbors=5)

knn_scaled_model.fit(X_train_scaled, y_train)

# Make predictions
y_pred_knn_scaled = knn_scaled_model.predict(X_test_scaled)

In [66]:
# Evaluate scaled KNN
mae_knn_scaled = mean_absolute_error(y_test, y_pred_knn_scaled)
rmse_knn_scaled = np.sqrt(
    mean_squared_error(y_test, y_pred_knn_scaled)
)

mae_knn_scaled, rmse_knn_scaled

(9.731111111111112, np.float64(12.473064312081988))

### Scaled KNN Comparison

After standardizing the input features, the KNN regression model improved:

- MAE: 9.73
- RMSE: 12.47

This is better than the unscaled KNN model, confirming that feature scaling is important for distance-based methods.

However, Linear Regression still achieves the best performance.

## Model Comparison Summary

The tested regression models are compared using the same chronological train-test split and the same evaluation metrics.

Lower MAE and RMSE values indicate better predictive performance.

In [67]:
# Compare all tested models
model_comparison = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "KNN",
        "KNN Scaled",
        "Decision Tree"
    ],
    "MAE": [
        mae_goalies,
        mae_knn,
        mae_knn_scaled,
        mae_tree
    ],
    "RMSE": [
        rmse_goalies,
        rmse_knn,
        rmse_knn_scaled,
        rmse_tree
    ]
})

model_comparison.sort_values("MAE")

,Model,MAE,RMSE
0,Linear Regression,8.807569,11.160234
2,KNN Scaled,9.731111,12.473064
1,KNN,10.697778,13.113250
3,Decision Tree,12.399535,15.561684


### Best Model

Among the tested models, Linear Regression achieved the lowest MAE and RMSE.

Therefore, Linear Regression is selected as the best model so far for predicting next-season team points.

## Final Ranking Prediction

The best-performing model, Linear Regression, is used to predict team points.

The predicted points are then used to order the teams and obtain a predicted final regular-season ranking.

In [68]:
# Create final ranking results using the best model
final_ranking = test_data[
    ["year", "tmID", "name", "Pts"]
].copy()

final_ranking["Predicted_Pts"] = y_pred_goalies

final_ranking["Actual_Rank"] = (
    final_ranking
    .groupby("year")["Pts"]
    .rank(method="min", ascending=False)
    .astype(int)
)

final_ranking["Predicted_Rank"] = (
    final_ranking
    .groupby("year")["Predicted_Pts"]
    .rank(method="min", ascending=False)
    .astype(int)
)

In [69]:
final_ranking[
    final_ranking["year"] == 2009
][[
    "Predicted_Rank",
    "name",
    "Predicted_Pts",
    "Actual_Rank",
    "Pts"
]].sort_values("Predicted_Rank")

,Predicted_Rank,name,Predicted_Pts,Actual_Rank,Pts
322,1,San Jose Sharks,105.893324,2,113
134,2,Detroit Red Wings,105.322868,7,102
224,3,New Jersey Devils,102.563961,5,103
92,4,Chicago Blackhawks,102.210383,3,112
27,5,Boston Bruins,101.000256,14,91
252,6,New York Rangers,98.792859,21,87
392,7,Washington Capitals,96.955917,1,121
69,8,Carolina Hurricanes,96.523339,24,80
41,9,Buffalo Sabres,94.493206,10,100
308,10,Pittsburgh Penguins,94.302516,8,101


## Ridge Regression

Ridge Regression is tested as an alternative to ordinary Linear Regression.

This method introduces a regularization penalty and can be useful when several predictor variables are correlated.

In [70]:
from sklearn.linear_model import Ridge

# Create and train the Ridge Regression model
ridge_model = Ridge(alpha=1.0)

ridge_model.fit(X_train_scaled, y_train)

# Make predictions
y_pred_ridge = ridge_model.predict(X_test_scaled)

In [71]:
# Evaluate the Ridge Regression model
mae_ridge = mean_absolute_error(y_test, y_pred_ridge)
rmse_ridge = np.sqrt(
    mean_squared_error(y_test, y_pred_ridge)
)

mae_ridge, rmse_ridge

(8.706417755543496, np.float64(11.129737728212978))

### Ridge Regression Comparison

The Ridge Regression model achieved:

- MAE: 8.71
- RMSE: 11.13

This slightly improves the previous best Linear Regression model.

The result suggests that regularization is useful when working with correlated predictors.

## Ridge Hyperparameter Tuning

The Ridge regularization parameter `alpha` is tuned using a chronological validation split inside the training data.

This avoids using the final test set during model selection.

In [72]:
# Create a chronological validation split inside the training data
inner_train = train_data[train_data["year"] <= 2003]
validation = train_data[train_data["year"] >= 2005]

In [73]:
# Prepare inner training and validation sets
X_inner_train = inner_train[features_with_goalies]
y_inner_train = inner_train["Pts"]

X_validation = validation[features_with_goalies]
y_validation = validation["Pts"]

In [74]:
# Scale features using only the inner training set
ridge_scaler = StandardScaler()

X_inner_train_scaled = ridge_scaler.fit_transform(X_inner_train)
X_validation_scaled = ridge_scaler.transform(X_validation)

In [75]:
# Test different alpha values
alpha_values = [0.01, 0.1, 1, 10, 100]

ridge_results = []

for alpha in alpha_values:
    model = Ridge(alpha=alpha)
    model.fit(X_inner_train_scaled, y_inner_train)

    validation_pred = model.predict(X_validation_scaled)

    mae = mean_absolute_error(y_validation, validation_pred)
    rmse = np.sqrt(mean_squared_error(y_validation, validation_pred))

    ridge_results.append({
        "alpha": alpha,
        "MAE": mae,
        "RMSE": rmse
    })

ridge_tuning = pd.DataFrame(ridge_results)

ridge_tuning

,alpha,MAE,RMSE
0,0.01,11.232558,14.434274
1,0.10,11.265175,14.448263
2,1.00,11.385419,14.502343
3,10.00,11.413153,14.518307
4,100.00,11.332088,14.414714


### Ridge Tuning Result

Among the tested values, `alpha = 0.01` achieved the lowest validation MAE and RMSE.

The differences between alpha values are relatively small, suggesting that model performance is not highly sensitive to this hyperparameter in the tested range.

Therefore, `alpha = 0.01` is selected for the final Ridge model.

In [76]:
# Scale the full training and test sets
final_ridge_scaler = StandardScaler()

X_train_scaled = final_ridge_scaler.fit_transform(X_train)
X_test_scaled = final_ridge_scaler.transform(X_test)

# Train the tuned Ridge model
final_ridge_model = Ridge(alpha=0.01)

final_ridge_model.fit(X_train_scaled, y_train)

# Make predictions
y_pred_ridge_tuned = final_ridge_model.predict(X_test_scaled)

In [77]:
# Evaluate the tuned Ridge model
mae_ridge_tuned = mean_absolute_error(y_test, y_pred_ridge_tuned)
rmse_ridge_tuned = np.sqrt(
    mean_squared_error(y_test, y_pred_ridge_tuned)
)

mae_ridge_tuned, rmse_ridge_tuned

(8.78795209440133, np.float64(11.144077466971456))

### Tuned Ridge Evaluation

The tuned Ridge model with `alpha = 0.01` achieved:

- MAE: 8.79
- RMSE: 11.14

This result is slightly worse than the Ridge model with `alpha = 1.0`.

Therefore, the original Ridge model with `alpha = 1.0` remains the best Ridge configuration on the final test set.

## Random Forest Regression

A Random Forest regression model is tested using the same training and test sets.

This ensemble method combines multiple decision trees and can capture non-linear relationships and interactions between predictor variables.

In [78]:
from sklearn.ensemble import RandomForestRegressor

# Create the Random Forest regression model
rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=None,
    random_state=42
)

# Train the model
rf_model.fit(X_train, y_train)

# Make predictions
y_pred_rf = rf_model.predict(X_test)

In [79]:
# Evaluate the Random Forest model
mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(
    mean_squared_error(y_test, y_pred_rf)
)

mae_rf, rmse_rf

(8.916666666666666, np.float64(11.301136348959682))

### Random Forest Comparison

The Random Forest regression model achieved:

- MAE: 8.92
- RMSE: 11.30

These results are worse than the Ridge Regression model.

Therefore, Random Forest does not improve predictive performance with the current configuration.

## Ranking-Based Model Comparison

Since the objective of Question 1 is to predict the final regular-season ranking, the models are also evaluated directly on ranking quality.

For each model, predicted points are converted into a predicted ranking and compared with the actual ranking.

In [80]:
# Function to evaluate ranking quality
def evaluate_ranking(test_data, predictions):
    ranking = test_data[
        ["year", "tmID", "name", "Pts"]
    ].copy()

    ranking["Predicted_Pts"] = predictions

    ranking["Actual_Rank"] = (
        ranking.groupby("year")["Pts"]
        .rank(method="min", ascending=False)
    )

    ranking["Predicted_Rank"] = (
        ranking.groupby("year")["Predicted_Pts"]
        .rank(method="min", ascending=False)
    )

    ranking["Absolute_Rank_Error"] = (
        ranking["Actual_Rank"] - ranking["Predicted_Rank"]
    ).abs()

    mean_rank_error = ranking["Absolute_Rank_Error"].mean()

    rank_correlation = ranking[
        ["Actual_Rank", "Predicted_Rank"]
    ].corr().iloc[0, 1]

    return mean_rank_error, rank_correlation

In [81]:
# Evaluate ranking performance for each model
linear_rank_error, linear_rank_corr = evaluate_ranking(
    test_data, y_pred_goalies
)

ridge_rank_error, ridge_rank_corr = evaluate_ranking(
    test_data, y_pred_ridge
)

rf_rank_error, rf_rank_corr = evaluate_ranking(
    test_data, y_pred_rf
)

ranking_comparison = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Ridge Regression",
        "Random Forest"
    ],
    "Mean Rank Error": [
        linear_rank_error,
        ridge_rank_error,
        rf_rank_error
    ],
    "Rank Correlation": [
        linear_rank_corr,
        ridge_rank_corr,
        rf_rank_corr
    ]
})

ranking_comparison.sort_values("Mean Rank Error")

,Model,Mean Rank Error,Rank Correlation
0,Linear Regression,6.755556,0.487525
2,Random Forest,6.800000,0.484992
1,Ridge Regression,6.888889,0.484545


### Ranking-Based Model Selection

When the models are evaluated directly on ranking quality, Linear Regression performs best.

It achieved:

- Mean Rank Error: 6.76 positions
- Rank Correlation: 0.49

Although Ridge Regression achieved slightly better MAE and RMSE for point prediction, Linear Regression produced the best final team ordering.

Therefore, Linear Regression is currently the preferred model for Question 1.

## Trend Features

The current predictors mainly describe the immediately previous season.

Additional trend features are created to capture recent multi-season performance patterns, which may help identify teams that are improving or declining over time.

In [82]:
# Sort by franchise and season
recent_teams = recent_teams.sort_values(["franchID", "year"])

# Points from two seasons ago
recent_teams["previous_2_Pts"] = (
    recent_teams.groupby("franchID")["Pts"].shift(2)
)

# Change in points between the two previous seasons
recent_teams["previous_Pts_change"] = (
    recent_teams["previous_Pts"] - recent_teams["previous_2_Pts"]
)

In [83]:
recent_teams[
    recent_teams["franchID"] == "ANA"
][[
    "year",
    "Pts",
    "previous_Pts",
    "previous_2_Pts",
    "previous_Pts_change"
]]

,year,Pts,previous_Pts,previous_2_Pts,previous_Pts_change
0,1995,78,NaN,NaN,NaN
1,1996,85,78.0,NaN,NaN
2,1997,65,85.0,78.0,7.0
3,1998,83,65.0,85.0,-20.0
4,1999,83,83.0,65.0,18.0
5,2000,66,83.0,83.0,0.0
6,2001,69,66.0,83.0,-17.0
7,2002,95,69.0,66.0,3.0
8,2003,76,95.0,69.0,26.0
9,2005,98,76.0,95.0,-19.0


In [84]:
model_data = recent_teams.dropna(
    subset=[
        "previous_W",
        "previous_L",
        "previous_GF",
        "previous_GA",
        "previous_Pts",
        "previous_player_G",
        "previous_player_A",
        "previous_player_Pts",
        "previous_player_SOG",
        "previous_goalie_W",
        "previous_goalie_L",
        "previous_goalie_SHO",
        "previous_goalie_GA",
        "previous_goalie_SA",
        "previous_2_Pts",
        "previous_Pts_change"
    ]
)

model_data.shape

(343, 52)

## Linear Regression with Trend Features

The Linear Regression model is retrained with additional trend features based on the previous two seasons.

The goal is to determine whether recent performance direction helps improve ranking prediction.

In [85]:
features_with_trends = [
    "previous_W",
    "previous_L",
    "previous_GF",
    "previous_GA",
    "previous_Pts",
    "previous_player_G",
    "previous_player_A",
    "previous_player_Pts",
    "previous_player_SOG",
    "previous_goalie_W",
    "previous_goalie_L",
    "previous_goalie_SHO",
    "previous_goalie_GA",
    "previous_goalie_SA",
    "previous_2_Pts",
    "previous_Pts_change"
]

In [86]:
# Split the data chronologically
train_data = model_data[model_data["year"] <= 2006]
test_data = model_data[model_data["year"] >= 2007]

In [87]:
X_train = train_data[features_with_trends]
y_train = train_data["Pts"]

X_test = test_data[features_with_trends]
y_test = test_data["Pts"]

In [88]:
# Train Linear Regression with trend features
trend_model = LinearRegression()

trend_model.fit(X_train, y_train)

y_pred_trend = trend_model.predict(X_test)

In [89]:
mae_trend = mean_absolute_error(y_test, y_pred_trend)
rmse_trend = np.sqrt(
    mean_squared_error(y_test, y_pred_trend)
)

mae_trend, rmse_trend

(8.992113244050875, np.float64(11.416967505725388))

In [90]:
trend_rank_error, trend_rank_corr = evaluate_ranking(
    test_data,
    y_pred_trend
)

trend_rank_error, trend_rank_corr

(np.float64(6.977777777777778), np.float64(0.4607011624965417))

### Trend Features Evaluation

The additional trend features did not improve model performance.

The model achieved:

- Mean Rank Error: 6.98 positions
- Rank Correlation: 0.46

Both results are worse than the previous Linear Regression model.

Therefore, these trend features are not retained in the final feature set.

## Reduced Feature Set

Several predictor variables contain overlapping information.

A reduced feature set is tested to determine whether removing redundant variables improves generalization and ranking performance.

In [91]:
model_data = recent_teams.dropna(
    subset=[
        "previous_W",
        "previous_L",
        "previous_GF",
        "previous_GA",
        "previous_Pts",
        "previous_player_G",
        "previous_player_A",
        "previous_player_Pts",
        "previous_player_SOG",
        "previous_goalie_W",
        "previous_goalie_L",
        "previous_goalie_SHO",
        "previous_goalie_GA",
        "previous_goalie_SA"
    ]
)

In [92]:
reduced_features = [
    "previous_Pts",
    "previous_GF",
    "previous_GA",
    "previous_player_Pts",
    "previous_player_SOG",
    "previous_goalie_SHO",
    "previous_goalie_GA",
    "previous_goalie_SA"
]

In [93]:
train_data = model_data[model_data["year"] <= 2006]
test_data = model_data[model_data["year"] >= 2007]

X_train = train_data[reduced_features]
y_train = train_data["Pts"]

X_test = test_data[reduced_features]
y_test = test_data["Pts"]

reduced_model = LinearRegression()
reduced_model.fit(X_train, y_train)

y_pred_reduced = reduced_model.predict(X_test)

In [94]:
mae_reduced = mean_absolute_error(y_test, y_pred_reduced)
rmse_reduced = np.sqrt(
    mean_squared_error(y_test, y_pred_reduced)
)

mae_reduced, rmse_reduced

(8.669806945503494, np.float64(10.959421429363152))

In [95]:
reduced_rank_error, reduced_rank_corr = evaluate_ranking(
    test_data,
    y_pred_reduced
)

reduced_rank_error, reduced_rank_corr

(np.float64(6.911111111111111), np.float64(0.48260737983019586))

### Reduced Feature Evaluation

The reduced feature set improved point prediction:

- MAE: 8.67
- RMSE: 10.96

However, ranking performance became slightly worse:

- Mean Rank Error: 6.91 positions
- Rank Correlation: 0.48

Therefore, although feature reduction improved point prediction, it did not improve the final team ordering.

The original Linear Regression model remains the best model for ranking prediction.

## Lasso Regression

Lasso Regression is tested as a regularized alternative to ordinary Linear Regression.

Lasso can reduce the influence of less useful predictors and may produce a simpler model by shrinking some coefficients toward zero.

In [96]:
features_with_goalies = [
    "previous_W",
    "previous_L",
    "previous_GF",
    "previous_GA",
    "previous_Pts",
    "previous_player_G",
    "previous_player_A",
    "previous_player_Pts",
    "previous_player_SOG",
    "previous_goalie_W",
    "previous_goalie_L",
    "previous_goalie_SHO",
    "previous_goalie_GA",
    "previous_goalie_SA"
]

In [97]:
train_data = model_data[model_data["year"] <= 2006]
test_data = model_data[model_data["year"] >= 2007]

X_train = train_data[features_with_goalies]
y_train = train_data["Pts"]

X_test = test_data[features_with_goalies]
y_test = test_data["Pts"]

In [98]:
from sklearn.preprocessing import StandardScaler

lasso_scaler = StandardScaler()

X_train_scaled = lasso_scaler.fit_transform(X_train)
X_test_scaled = lasso_scaler.transform(X_test)

In [99]:
from sklearn.linear_model import Lasso

# Train the Lasso Regression model
lasso_model = Lasso(
    alpha=0.1,
    max_iter=10000
)

lasso_model.fit(X_train_scaled, y_train)

# Make predictions
y_pred_lasso = lasso_model.predict(X_test_scaled)

In [100]:
mae_lasso = mean_absolute_error(y_test, y_pred_lasso)
rmse_lasso = np.sqrt(
    mean_squared_error(y_test, y_pred_lasso)
)

mae_lasso, rmse_lasso

(8.840837503659607, np.float64(11.23067298450185))

In [101]:
lasso_rank_error, lasso_rank_corr = evaluate_ranking(
    test_data,
    y_pred_lasso
)

lasso_rank_error, lasso_rank_corr

(np.float64(6.8), np.float64(0.500937071884886))

### Lasso Ranking Evaluation

The Lasso model achieved:

- Mean Rank Error: 6.80 positions
- Rank Correlation: 0.50

Although its mean ranking error is slightly worse than Linear Regression, its ranking correlation is slightly higher.

Therefore, Lasso remains a promising model and its regularization parameter should be tuned before making a final comparison.

In [102]:
# Create chronological validation split
inner_train = train_data[train_data["year"] <= 2003]
validation = train_data[train_data["year"] >= 2005]

X_inner_train = inner_train[features_with_goalies]
y_inner_train = inner_train["Pts"]

X_validation = validation[features_with_goalies]
y_validation = validation["Pts"]

In [103]:
# Scale inner training and validation sets
lasso_scaler = StandardScaler()

X_inner_train_scaled = lasso_scaler.fit_transform(X_inner_train)
X_validation_scaled = lasso_scaler.transform(X_validation)

In [104]:
alpha_values = [0.001, 0.01, 0.05, 0.1, 0.5, 1]

lasso_results = []

for alpha in alpha_values:
    model = Lasso(alpha=alpha, max_iter=10000)

    model.fit(X_inner_train_scaled, y_inner_train)

    validation_pred = model.predict(X_validation_scaled)

    mae = mean_absolute_error(y_validation, validation_pred)
    rmse = np.sqrt(mean_squared_error(y_validation, validation_pred))

    lasso_results.append({
        "alpha": alpha,
        "MAE": mae,
        "RMSE": rmse
    })

lasso_tuning = pd.DataFrame(lasso_results)

lasso_tuning

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:842: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.189456e+01, tolerance: 5.209e+00
  model = cd_fast.enet_coordinate_descent(


,alpha,MAE,RMSE
0,0.001,11.220699,14.424914
1,0.010,11.347295,14.481789
2,0.050,11.390705,14.512601
3,0.100,11.421371,14.566962
4,0.500,11.491318,14.622942
5,1.000,11.612702,14.652629


### Lasso Tuning Result

The best validation performance was obtained with `alpha = 0.001`.

Lower alpha values apply less regularization, making the model behave more similarly to ordinary Linear Regression.

A convergence warning was observed for very small alpha values, so the final model is trained with a higher iteration limit.

In [105]:
# Scale the full training and test sets
final_lasso_scaler = StandardScaler()

X_train_scaled = final_lasso_scaler.fit_transform(X_train)
X_test_scaled = final_lasso_scaler.transform(X_test)

# Train the tuned Lasso model
final_lasso_model = Lasso(
    alpha=0.001,
    max_iter=100000
)

final_lasso_model.fit(X_train_scaled, y_train)

# Make predictions
y_pred_lasso_tuned = final_lasso_model.predict(X_test_scaled)

In [106]:
mae_lasso_tuned = mean_absolute_error(
    y_test,
    y_pred_lasso_tuned
)

rmse_lasso_tuned = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred_lasso_tuned
    )
)

mae_lasso_tuned, rmse_lasso_tuned

(8.786520655755018, np.float64(11.145298901541604))

In [107]:
lasso_tuned_rank_error, lasso_tuned_rank_corr = evaluate_ranking(
    test_data,
    y_pred_lasso_tuned
)

lasso_tuned_rank_error, lasso_tuned_rank_corr

(np.float64(6.777777777777778), np.float64(0.4861839051091597))

### Tuned Lasso Ranking Evaluation

The tuned Lasso model achieved:

- Mean Rank Error: 6.78 positions
- Rank Correlation: 0.49

These results are very similar to Linear Regression, but do not improve the final ranking prediction.

Therefore, Linear Regression remains the best model for Question 1 so far.

## Support Vector Regression

Support Vector Regression is tested as a non-linear alternative to the previous regression models.

The model is evaluated using scaled features because SVR is sensitive to the scale of the input variables.

In [108]:
features_with_goalies = [
    "previous_W",
    "previous_L",
    "previous_GF",
    "previous_GA",
    "previous_Pts",
    "previous_player_G",
    "previous_player_A",
    "previous_player_Pts",
    "previous_player_SOG",
    "previous_goalie_W",
    "previous_goalie_L",
    "previous_goalie_SHO",
    "previous_goalie_GA",
    "previous_goalie_SA"
]

In [109]:
train_data = model_data[model_data["year"] <= 2006]
test_data = model_data[model_data["year"] >= 2007]

X_train = train_data[features_with_goalies]
y_train = train_data["Pts"]

X_test = test_data[features_with_goalies]
y_test = test_data["Pts"]

svr_scaler = StandardScaler()

X_train_scaled = svr_scaler.fit_transform(X_train)
X_test_scaled = svr_scaler.transform(X_test)

In [110]:
from sklearn.svm import SVR

# Train SVR with RBF kernel
svr_model = SVR(
    kernel="rbf",
    C=10,
    epsilon=0.1
)

svr_model.fit(X_train_scaled, y_train)

# Make predictions
y_pred_svr = svr_model.predict(X_test_scaled)

In [111]:
mae_svr = mean_absolute_error(y_test, y_pred_svr)
rmse_svr = np.sqrt(
    mean_squared_error(y_test, y_pred_svr)
)

mae_svr, rmse_svr

(8.989231924908186, np.float64(11.398804098101149))

### SVR Point Evaluation

The RBF Support Vector Regression model achieved:

- MAE: 8.99
- RMSE: 11.40

These results are worse than the best linear models.

Therefore, this SVR configuration does not improve point prediction.

In [112]:
svr_rank_error, svr_rank_corr = evaluate_ranking(
    test_data,
    y_pred_svr
)

svr_rank_error, svr_rank_corr

(np.float64(6.6), np.float64(0.49885076547215695))

### SVR Ranking Evaluation

The Support Vector Regression model achieved:

- Mean Rank Error: 6.60 positions
- Rank Correlation: 0.50

Although its point prediction errors are slightly worse than some linear models, SVR produces the best ranking performance obtained so far.

Therefore, SVR becomes the current best model for Question 1.

## SVR Hyperparameter Tuning

The SVR hyperparameters are tuned using a chronological validation split inside the training data.

The goal is to improve ranking performance without using the final test set during model selection.

In [113]:
# Create chronological validation split
inner_train = train_data[train_data["year"] <= 2003]
validation = train_data[train_data["year"] >= 2005]

X_inner_train = inner_train[features_with_goalies]
y_inner_train = inner_train["Pts"]

X_validation = validation[features_with_goalies]
y_validation = validation["Pts"]

In [114]:
# Scale inner training and validation sets
svr_tuning_scaler = StandardScaler()

X_inner_train_scaled = svr_tuning_scaler.fit_transform(X_inner_train)
X_validation_scaled = svr_tuning_scaler.transform(X_validation)

In [115]:
C_values = [0.1, 1, 10, 100]
epsilon_values = [0.1, 0.5, 1.0]

svr_results = []

for C in C_values:
    for epsilon in epsilon_values:
        model = SVR(
            kernel="rbf",
            C=C,
            epsilon=epsilon
        )

        model.fit(X_inner_train_scaled, y_inner_train)

        validation_pred = model.predict(X_validation_scaled)

        mae = mean_absolute_error(y_validation, validation_pred)
        rmse = np.sqrt(
            mean_squared_error(y_validation, validation_pred)
        )

        rank_error, rank_corr = evaluate_ranking(
            validation,
            validation_pred
        )

        svr_results.append({
            "C": C,
            "epsilon": epsilon,
            "MAE": mae,
            "RMSE": rmse,
            "Rank_Error": rank_error,
            "Rank_Correlation": rank_corr
        })

svr_tuning = pd.DataFrame(svr_results)

svr_tuning.sort_values("Rank_Error")

,C,epsilon,MAE,RMSE,Rank_Error,Rank_Correlation
3,1.0,0.1,12.029877,14.624754,6.383333,0.545614
4,1.0,0.5,12.097987,14.677197,6.383333,0.545394
5,1.0,1.0,12.055034,14.665121,6.383333,0.545394
2,0.1,1.0,13.389814,15.963669,6.683333,0.512752
0,0.1,0.1,13.291983,15.891080,6.716667,0.503331
1,0.1,0.5,13.325175,15.917470,6.750000,0.504865
7,10.0,0.5,11.928114,14.522340,6.750000,0.527211
6,10.0,0.1,11.975119,14.578953,6.783333,0.525239
8,10.0,1.0,11.834478,14.446894,6.783333,0.523706
11,100.0,1.0,12.087630,15.204942,7.016667,0.470689


### SVR Tuning Result

The best validation ranking performance was obtained with:

- `C = 1`
- `epsilon = 0.1`
- RBF kernel

This configuration achieved a mean rank error of approximately 6.38 positions and a rank correlation of approximately 0.55 on the validation set.

Therefore, these hyperparameters are selected for the final SVR model.

In [116]:
# Scale the full training and test sets
final_svr_scaler = StandardScaler()

X_train_scaled = final_svr_scaler.fit_transform(X_train)
X_test_scaled = final_svr_scaler.transform(X_test)

# Train the tuned SVR model
final_svr_model = SVR(
    kernel="rbf",
    C=1,
    epsilon=0.1
)

final_svr_model.fit(X_train_scaled, y_train)

# Make predictions
y_pred_svr_tuned = final_svr_model.predict(X_test_scaled)

In [117]:
svr_tuned_rank_error, svr_tuned_rank_corr = evaluate_ranking(
    test_data,
    y_pred_svr_tuned
)

svr_tuned_rank_error, svr_tuned_rank_corr

(np.float64(6.5777777777777775), np.float64(0.507792078669567))

### Final Tuned SVR Evaluation

The tuned SVR model achieved:

- Mean Rank Error: 6.58 positions
- Rank Correlation: 0.51

This slightly improves the previous SVR configuration and gives the best ranking performance obtained so far.

Therefore, the tuned SVR model is currently the best model for Question 1.

## Final Model Comparison

The main regression models are compared using both point prediction metrics and ranking metrics.

Since Question 1 focuses on the final regular-season ranking, ranking error and ranking correlation are especially important for model selection.

In [118]:
final_model_comparison = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Ridge Regression",
        "Lasso",
        "Random Forest",
        "SVR",
        "SVR Tuned"
    ],
    "MAE": [
        mae_goalies,
        mae_ridge,
        mae_lasso,
        mae_rf,
        mae_svr,
        mean_absolute_error(y_test, y_pred_svr_tuned)
    ],
    "RMSE": [
        rmse_goalies,
        rmse_ridge,
        rmse_lasso,
        rmse_rf,
        rmse_svr,
        np.sqrt(mean_squared_error(y_test, y_pred_svr_tuned))
    ],
    "Mean Rank Error": [
        linear_rank_error,
        ridge_rank_error,
        lasso_rank_error,
        rf_rank_error,
        svr_rank_error,
        svr_tuned_rank_error
    ],
    "Rank Correlation": [
        linear_rank_corr,
        ridge_rank_corr,
        lasso_rank_corr,
        rf_rank_corr,
        svr_rank_corr,
        svr_tuned_rank_corr
    ]
})

final_model_comparison.sort_values("Mean Rank Error")

,Model,MAE,RMSE,Mean Rank Error,Rank Correlation
5,SVR Tuned,8.590012,11.095322,6.577778,0.507792
4,SVR,8.989232,11.398804,6.600000,0.498851
0,Linear Regression,8.807569,11.160234,6.755556,0.487525
2,Lasso,8.840838,11.230673,6.800000,0.500937
3,Random Forest,8.916667,11.301136,6.800000,0.484992
1,Ridge Regression,8.706418,11.129738,6.888889,0.484545


### Final Model Selection

Among all tested models, the tuned Support Vector Regression model achieved the best overall performance:

- MAE: 8.59
- RMSE: 11.10
- Mean Rank Error: 6.58 positions
- Rank Correlation: 0.51

The tuned SVR model improves both point prediction and ranking performance compared with the other tested models.

Therefore, it is selected as the current best model for Question 1.

In [119]:
# Create ranking results for the tuned SVR model
svr_final_results = test_data[
    ["year", "tmID", "name", "Pts"]
].copy()

svr_final_results["Predicted_Pts"] = y_pred_svr_tuned

svr_final_results["Actual_Rank"] = (
    svr_final_results
    .groupby("year")["Pts"]
    .rank(method="min", ascending=False)
)

svr_final_results["Predicted_Rank"] = (
    svr_final_results
    .groupby("year")["Predicted_Pts"]
    .rank(method="min", ascending=False)
)

svr_final_results["Absolute_Rank_Error"] = (
    svr_final_results["Actual_Rank"]
    - svr_final_results["Predicted_Rank"]
).abs()

In [120]:
# Mean ranking error for each test season
svr_final_results.groupby("year")["Absolute_Rank_Error"].mean()

year
2007    6.633333
2008    6.666667
2009    6.433333
Name: Absolute_Rank_Error, dtype: float64

### Ranking Performance by Season

The tuned SVR model shows consistent ranking performance across the three test seasons:

- 2007: mean rank error of 6.63 positions
- 2008: mean rank error of 6.67 positions
- 2009: mean rank error of 6.43 positions

This indicates that the model's performance is relatively stable across the test period and is not dominated by a single unusually good or bad season.

In [121]:
# Show the largest ranking errors for the tuned SVR model
svr_final_results.sort_values(
    "Absolute_Rank_Error",
    ascending=False
)[[
    "year",
    "name",
    "Pts",
    "Predicted_Pts",
    "Actual_Rank",
    "Predicted_Rank",
    "Absolute_Rank_Error"
]].head(15)

,year,name,Pts,Predicted_Pts,Actual_Rank,Predicted_Rank,Absolute_Rank_Error
294,2009,Phoenix Coyotes,107,79.825595,4.0,27.0,23.0
105,2008,Colorado Avalanche,69,93.818802,28.0,6.0,22.0
278,2007,Philadelphia Flyers,95,77.670955,10.0,30.0,20.0
119,2008,Dallas Stars,83,94.863105,22.0,3.0,19.0
197,2007,Montreal Canadiens,104,85.096713,3.0,21.0,18.0
69,2009,Carolina Hurricanes,80,93.504898,24.0,6.0,18.0
390,2007,Washington Capitals,94,78.292416,12.0,28.0,16.0
106,2009,Colorado Avalanche,95,79.646986,12.0,28.0,16.0
26,2008,Boston Bruins,116,88.641585,2.0,18.0,16.0
176,2009,Los Angeles Kings,101,84.365732,8.0,23.0,15.0


### Largest Ranking Errors

The largest ranking errors still occur mainly for teams whose performance changed substantially between seasons.

Examples include Phoenix in 2009, Colorado in 2008, Philadelphia in 2007, and Boston in 2008.

This indicates that even the tuned SVR model has difficulty predicting sudden improvements or declines based only on previous-season information.

## Roster Continuity Features

Large year-to-year performance changes may be related to changes in team composition.

A roster continuity feature is created to measure how many players remain with the same franchise from one season to the next.

In [122]:
# Add franchise identifiers to the scoring data
scoring_with_franchise = recent_scoring.merge(
    recent_teams[["year", "tmID", "franchID"]].drop_duplicates(),
    on=["year", "tmID"],
    how="left"
)

In [123]:
scoring_with_franchise[
    ["year", "tmID", "franchID", "playerID"]
].head()

,year,tmID,franchID,playerID
0,1997,ANA,ANA,aaltoan01
1,1998,ANA,ANA,aaltoan01
2,1999,ANA,ANA,aaltoan01
3,2000,ANA,ANA,aaltoan01
4,2007,DET,DET,abdelju01


In [124]:
# Create the set of players for each franchise-season
rosters = (
    scoring_with_franchise
    .dropna(subset=["franchID"])
    .groupby(["franchID", "year"])["playerID"]
    .apply(set)
    .reset_index(name="players")
)

rosters.head()

,franchID,year,players
0,ANA,1995,"{prongse01, lambede01, hicksal01, butsavi01, d..."
1,ANA,1996,"{prongse01, trebida01, daignjj01, nikulig01, h..."
2,ANA,1997,"{prongse01, trebida01, crowlmi01, moroma01 , d..."
3,ANA,1998,"{trebida01, crowlmi01, mcinnma01, trepapa01, s..."
4,ANA,1999,"{balmoma01, hrkacto01, mcinnma01, vishnvi01, t..."


In [126]:
# Sort by franchise and season
rosters = rosters.sort_values(["franchID", "year"])

# Get the previous season roster
rosters["previous_players"] = (
    rosters.groupby("franchID")["players"].shift(1)
)

In [127]:
# Calculate roster continuity between consecutive seasons
def calculate_roster_continuity(row):
    if not isinstance(row["previous_players"], set):
        return np.nan

    returning_players = len(
        row["players"].intersection(row["previous_players"])
    )

    return returning_players / len(row["previous_players"])

rosters["roster_continuity"] = rosters.apply(
    calculate_roster_continuity,
    axis=1
)

In [128]:
rosters[
    rosters["franchID"] == "ANA"
][[
    "franchID",
    "year",
    "roster_continuity"
]]

,franchID,year,roster_continuity
0,ANA,1995,NaN
1,ANA,1996,0.452381
2,ANA,1997,0.675000
3,ANA,1998,0.422222
4,ANA,1999,0.714286
5,ANA,2000,0.562500
6,ANA,2001,0.600000
7,ANA,2002,0.516129
8,ANA,2003,0.606061
9,ANA,2005,0.371429


In [129]:
# Use only roster continuity known before the predicted season
rosters["previous_roster_continuity"] = (
    rosters.groupby("franchID")["roster_continuity"].shift(1)
)

In [130]:
rosters[
    rosters["franchID"] == "ANA"
][[
    "year",
    "roster_continuity",
    "previous_roster_continuity"
]]

,year,roster_continuity,previous_roster_continuity
0,1995,NaN,NaN
1,1996,0.452381,NaN
2,1997,0.675000,0.452381
3,1998,0.422222,0.675000
4,1999,0.714286,0.422222
5,2000,0.562500,0.714286
6,2001,0.600000,0.562500
7,2002,0.516129,0.600000
8,2003,0.606061,0.516129
9,2005,0.371429,0.606061


In [131]:
# Merge previous roster continuity with team data
recent_teams = recent_teams.merge(
    rosters[[
        "franchID",
        "year",
        "previous_roster_continuity"
    ]],
    on=["franchID", "year"],
    how="left"
)

In [132]:
recent_teams[
    ["year", "tmID", "name", "previous_roster_continuity"]
].head(15)

,year,tmID,name,previous_roster_continuity
0,1995,ANA,Mighty Ducks of Anaheim,NaN
1,1996,ANA,Mighty Ducks of Anaheim,NaN
2,1997,ANA,Mighty Ducks of Anaheim,0.452381
3,1998,ANA,Mighty Ducks of Anaheim,0.675000
4,1999,ANA,Mighty Ducks of Anaheim,0.422222
5,2000,ANA,Mighty Ducks of Anaheim,0.714286
6,2001,ANA,Mighty Ducks of Anaheim,0.562500
7,2002,ANA,Mighty Ducks of Anaheim,0.600000
8,2003,ANA,Mighty Ducks of Anaheim,0.516129
9,2005,ANA,Mighty Ducks of Anaheim,0.606061


## SVR with Roster Continuity

The tuned SVR model is extended with a roster continuity feature.

This feature represents the proportion of players that remained with the same franchise between previous seasons and may help capture changes in team composition.

In [133]:
model_data_roster = recent_teams.dropna(
    subset=[
        "previous_W",
        "previous_L",
        "previous_GF",
        "previous_GA",
        "previous_Pts",
        "previous_player_G",
        "previous_player_A",
        "previous_player_Pts",
        "previous_player_SOG",
        "previous_goalie_W",
        "previous_goalie_L",
        "previous_goalie_SHO",
        "previous_goalie_GA",
        "previous_goalie_SA",
        "previous_roster_continuity"
    ]
)

model_data_roster.shape

(343, 53)

In [134]:
features_with_roster = [
    "previous_W",
    "previous_L",
    "previous_GF",
    "previous_GA",
    "previous_Pts",
    "previous_player_G",
    "previous_player_A",
    "previous_player_Pts",
    "previous_player_SOG",
    "previous_goalie_W",
    "previous_goalie_L",
    "previous_goalie_SHO",
    "previous_goalie_GA",
    "previous_goalie_SA",
    "previous_roster_continuity"
]

In [135]:
train_data_roster = model_data_roster[
    model_data_roster["year"] <= 2006
]

test_data_roster = model_data_roster[
    model_data_roster["year"] >= 2007
]

In [136]:
X_train_roster = train_data_roster[features_with_roster]
y_train_roster = train_data_roster["Pts"]

X_test_roster = test_data_roster[features_with_roster]
y_test_roster = test_data_roster["Pts"]

roster_scaler = StandardScaler()

X_train_roster_scaled = roster_scaler.fit_transform(X_train_roster)
X_test_roster_scaled = roster_scaler.transform(X_test_roster)

In [137]:
svr_roster_model = SVR(
    kernel="rbf",
    C=1,
    epsilon=0.1
)

svr_roster_model.fit(
    X_train_roster_scaled,
    y_train_roster
)

y_pred_svr_roster = svr_roster_model.predict(
    X_test_roster_scaled
)

In [138]:
roster_rank_error, roster_rank_corr = evaluate_ranking(
    test_data_roster,
    y_pred_svr_roster
)

roster_rank_error, roster_rank_corr

(np.float64(6.6), np.float64(0.522545245445293))

### Roster Continuity Evaluation

Adding previous roster continuity to the tuned SVR model resulted in:

- Mean Rank Error: 6.60 positions
- Rank Correlation: 0.52

The mean ranking error is slightly worse than the tuned SVR without this feature, but the ranking correlation improves.

This suggests that roster continuity contains useful information about the overall ordering of teams, although it does not reduce the average positional error.

In [139]:
# Use the same observations as the roster model
same_sample_data = model_data_roster.copy()

train_same = same_sample_data[
    same_sample_data["year"] <= 2006
]

test_same = same_sample_data[
    same_sample_data["year"] >= 2007
]

In [140]:
base_features = [
    "previous_W",
    "previous_L",
    "previous_GF",
    "previous_GA",
    "previous_Pts",
    "previous_player_G",
    "previous_player_A",
    "previous_player_Pts",
    "previous_player_SOG",
    "previous_goalie_W",
    "previous_goalie_L",
    "previous_goalie_SHO",
    "previous_goalie_GA",
    "previous_goalie_SA"
]

In [141]:
X_train_same = train_same[base_features]
y_train_same = train_same["Pts"]

X_test_same = test_same[base_features]
y_test_same = test_same["Pts"]

same_scaler = StandardScaler()

X_train_same_scaled = same_scaler.fit_transform(X_train_same)
X_test_same_scaled = same_scaler.transform(X_test_same)

In [142]:
svr_same_sample = SVR(
    kernel="rbf",
    C=1,
    epsilon=0.1
)

svr_same_sample.fit(
    X_train_same_scaled,
    y_train_same
)

y_pred_same_sample = svr_same_sample.predict(
    X_test_same_scaled
)

In [143]:
same_rank_error, same_rank_corr = evaluate_ranking(
    test_same,
    y_pred_same_sample
)

same_rank_error, same_rank_corr

(np.float64(6.622222222222222), np.float64(0.5125607790415188))

### Fair Roster Continuity Comparison

To ensure a fair comparison, the tuned SVR models were evaluated using the same observations.

Without roster continuity:

- Mean Rank Error: 6.62 positions
- Rank Correlation: 0.51

With roster continuity:

- Mean Rank Error: 6.60 positions
- Rank Correlation: 0.52

Therefore, roster continuity provides a small but consistent improvement in ranking prediction.

## Key Player Continuity

Roster continuity treats all players equally, but losing high-impact players may affect future performance more strongly.

A new feature is created to measure how much of the previous season's player production is retained by players who remain with the same franchise.

In [144]:
# Keep player points for each franchise-season
player_points = scoring_with_franchise[
    ["franchID", "year", "playerID", "Pts"]
].copy()

player_points = player_points.dropna(
    subset=["franchID", "Pts"]
)

In [145]:
# Aggregate player points within each franchise-season
player_points = (
    player_points
    .groupby(["franchID", "year", "playerID"])["Pts"]
    .sum()
    .reset_index()
)

In [146]:
player_points.head()

,franchID,year,playerID,Pts
0,ANA,1995,baumgke01,1.0
1,ANA,1995,butsavi01,1.0
2,ANA,1995,campbji01,5.0
3,ANA,1995,carnbpa01,18.0
4,ANA,1995,corkubo01,12.0


In [147]:
# Create a dictionary of players and their points for each franchise-season
season_player_points = (
    player_points
    .groupby(["franchID", "year"])
    .apply(
        lambda x: dict(zip(x["playerID"], x["Pts"]))
    )
    .reset_index(name="player_points_dict")
)

season_player_points.head()

,franchID,year,player_points_dict
0,ANA,1995,"{'baumgke01': 1.0, 'butsavi01': 1.0, 'campbji0..."
1,ANA,1996,"{'antossh01': 0.0, 'banhafr01': 0.0, 'baumgke0..."
2,ANA,1997,"{'aaltoan01': 0.0, 'antossh01': 1.0, 'askeyto0..."
3,ANA,1998,"{'aaltoan01': 8.0, 'crowlmi01': 5.0, 'cullema0..."
4,ANA,1999,"{'aaltoan01': 18.0, 'balmoma01': 1.0, 'banhafr..."


In [148]:
# Sort by franchise and season
season_player_points = season_player_points.sort_values(
    ["franchID", "year"]
)

# Get the previous season player-points dictionary
season_player_points["previous_player_points_dict"] = (
    season_player_points
    .groupby("franchID")["player_points_dict"]
    .shift(1)
)

In [149]:
# Calculate the proportion of previous-season points
# produced by players who remained with the franchise
def calculate_retained_production(row):
    previous = row["previous_player_points_dict"]
    current = row["player_points_dict"]

    if not isinstance(previous, dict):
        return np.nan

    total_previous_points = sum(previous.values())

    if total_previous_points == 0:
        return np.nan

    retained_points = sum(
        points
        for player, points in previous.items()
        if player in current
    )

    return retained_points / total_previous_points

In [150]:
season_player_points["retained_production"] = (
    season_player_points.apply(
        calculate_retained_production,
        axis=1
    )
)

In [151]:
season_player_points[
    season_player_points["franchID"] == "ANA"
][[
    "year",
    "retained_production"
]]

,year,retained_production
0,1995,NaN
1,1996,0.594976
2,1997,0.815710
3,1998,0.583174
4,1999,0.872566
5,2000,0.780405
6,2001,0.726027
7,2002,0.653191
8,2003,0.623656
9,2005,0.588353


In [152]:
# Use only retained production known before the predicted season
season_player_points["previous_retained_production"] = (
    season_player_points
    .groupby("franchID")["retained_production"]
    .shift(1)
)

In [153]:
season_player_points[
    season_player_points["franchID"] == "ANA"
][[
    "year",
    "retained_production",
    "previous_retained_production"
]]

,year,retained_production,previous_retained_production
0,1995,NaN,NaN
1,1996,0.594976,NaN
2,1997,0.815710,0.594976
3,1998,0.583174,0.815710
4,1999,0.872566,0.583174
5,2000,0.780405,0.872566
6,2001,0.726027,0.780405
7,2002,0.653191,0.726027
8,2003,0.623656,0.653191
9,2005,0.588353,0.623656


In [154]:
# Merge previous retained production with team data
recent_teams = recent_teams.merge(
    season_player_points[[
        "franchID",
        "year",
        "previous_retained_production"
    ]],
    on=["franchID", "year"],
    how="left"
)

In [155]:
recent_teams[
    ["year", "tmID", "name", "previous_retained_production"]
].head(15)

,year,tmID,name,previous_retained_production
0,1995,ANA,Mighty Ducks of Anaheim,NaN
1,1996,ANA,Mighty Ducks of Anaheim,NaN
2,1997,ANA,Mighty Ducks of Anaheim,0.594976
3,1998,ANA,Mighty Ducks of Anaheim,0.815710
4,1999,ANA,Mighty Ducks of Anaheim,0.583174
5,2000,ANA,Mighty Ducks of Anaheim,0.872566
6,2001,ANA,Mighty Ducks of Anaheim,0.780405
7,2002,ANA,Mighty Ducks of Anaheim,0.726027
8,2003,ANA,Mighty Ducks of Anaheim,0.653191
9,2005,ANA,Mighty Ducks of Anaheim,0.623656


In [158]:
# Merge previous retained production with team data
recent_teams = recent_teams.merge(
    season_player_points[[
        "franchID",
        "year",
        "previous_retained_production"
    ]],
    on=["franchID", "year"],
    how="left"
)

In [159]:
recent_teams[
    ["year", "tmID", "name", "previous_retained_production"]
].head(15)

,year,tmID,name,previous_retained_production
0,1995,ANA,Mighty Ducks of Anaheim,NaN
1,1996,ANA,Mighty Ducks of Anaheim,NaN
2,1997,ANA,Mighty Ducks of Anaheim,0.594976
3,1998,ANA,Mighty Ducks of Anaheim,0.815710
4,1999,ANA,Mighty Ducks of Anaheim,0.583174
5,2000,ANA,Mighty Ducks of Anaheim,0.872566
6,2001,ANA,Mighty Ducks of Anaheim,0.780405
7,2002,ANA,Mighty Ducks of Anaheim,0.726027
8,2003,ANA,Mighty Ducks of Anaheim,0.653191
9,2005,ANA,Mighty Ducks of Anaheim,0.623656


## SVR with Roster and Retained Production

The tuned SVR model is extended with two team-composition features:

- previous roster continuity
- previous retained offensive production

These features aim to capture how stable the team composition and offensive core were before the predicted season.

In [160]:
model_data_roster_prod = recent_teams.dropna(
    subset=[
        "previous_W",
        "previous_L",
        "previous_GF",
        "previous_GA",
        "previous_Pts",
        "previous_player_G",
        "previous_player_A",
        "previous_player_Pts",
        "previous_player_SOG",
        "previous_goalie_W",
        "previous_goalie_L",
        "previous_goalie_SHO",
        "previous_goalie_GA",
        "previous_goalie_SA",
        "previous_roster_continuity",
        "previous_retained_production"
    ]
)

model_data_roster_prod.shape

(343, 56)

In [161]:
features_roster_prod = [
    "previous_W",
    "previous_L",
    "previous_GF",
    "previous_GA",
    "previous_Pts",
    "previous_player_G",
    "previous_player_A",
    "previous_player_Pts",
    "previous_player_SOG",
    "previous_goalie_W",
    "previous_goalie_L",
    "previous_goalie_SHO",
    "previous_goalie_GA",
    "previous_goalie_SA",
    "previous_roster_continuity",
    "previous_retained_production"
]

In [162]:
train_rp = model_data_roster_prod[
    model_data_roster_prod["year"] <= 2006
]

test_rp = model_data_roster_prod[
    model_data_roster_prod["year"] >= 2007
]

X_train_rp = train_rp[features_roster_prod]
y_train_rp = train_rp["Pts"]

X_test_rp = test_rp[features_roster_prod]
y_test_rp = test_rp["Pts"]

rp_scaler = StandardScaler()

X_train_rp_scaled = rp_scaler.fit_transform(X_train_rp)
X_test_rp_scaled = rp_scaler.transform(X_test_rp)

In [163]:
svr_roster_prod_model = SVR(
    kernel="rbf",
    C=1,
    epsilon=0.1
)

svr_roster_prod_model.fit(
    X_train_rp_scaled,
    y_train_rp
)

y_pred_rp = svr_roster_prod_model.predict(
    X_test_rp_scaled
)

In [164]:
rp_rank_error, rp_rank_corr = evaluate_ranking(
    test_rp,
    y_pred_rp
)

rp_rank_error, rp_rank_corr

(np.float64(6.6), np.float64(0.5124117571548953))

### Retained Production Evaluation

Adding previous retained offensive production did not improve ranking performance.

The mean rank error remained at 6.60 positions, while rank correlation decreased from approximately 0.52 to 0.51.

Therefore, `previous_retained_production` is not retained in the final feature set.

## Main Goalie Continuity

Changes in the main goalkeeper may affect team defensive performance.

A feature is created to identify whether the goalkeeper with the most minutes played in the previous season remained the main goalkeeper in the following season.

In [165]:
# Identify the goalie with the most minutes played
main_goalies = (
    recent_goalies
    .sort_values(
        ["year", "tmID", "Min"],
        ascending=[True, True, False]
    )
    .groupby(["year", "tmID"])
    .first()
    .reset_index()
)

main_goalies = main_goalies[
    ["year", "tmID", "playerID", "Min"]
]

main_goalies.head()

,year,tmID,playerID,Min
0,1995,ANA,hebergu01,3326.0
1,1995,BOS,ranfobi01,2307.0
2,1995,BUF,hasekdo01,3417.0
3,1995,CAL,kiddtr01,2570.0
4,1995,CHI,belfoed01,2956.0


In [166]:
# Add franchise identifiers
main_goalies = main_goalies.merge(
    recent_teams[
        ["year", "tmID", "franchID"]
    ].drop_duplicates(),
    on=["year", "tmID"],
    how="left"
)

In [167]:
main_goalies[
    ["year", "tmID", "franchID", "playerID", "Min"]
].head(10)

,year,tmID,franchID,playerID,Min
0,1995,ANA,ANA,hebergu01,3326.0
1,1995,BOS,BOS,ranfobi01,2307.0
2,1995,BUF,BUF,hasekdo01,3417.0
3,1995,CAL,CAL,kiddtr01,2570.0
4,1995,CHI,CHI,belfoed01,2956.0
5,1995,COL,COL,roypa01,2305.0
6,1995,DAL,DAL,moogan01,2228.0
7,1995,DET,DET,osgooch01,2933.0
8,1995,EDM,EDM,ranfobi01,2015.0
9,1995,FLO,FLO,vanbijo01,3178.0


In [168]:
# Sort by franchise and season
main_goalies = main_goalies.sort_values(
    ["franchID", "year"]
)

In [169]:
# Get the previous season main goalie
main_goalies["previous_main_goalie"] = (
    main_goalies
    .groupby("franchID")["playerID"]
    .shift(1)
)

In [170]:
# Check whether the main goalie remained the same
main_goalies["main_goalie_continuity"] = (
    main_goalies["playerID"] ==
    main_goalies["previous_main_goalie"]
).astype(int)

In [171]:
main_goalies[
    main_goalies["franchID"] == "ANA"
][[
    "year",
    "playerID",
    "previous_main_goalie",
    "main_goalie_continuity"
]]

,year,playerID,previous_main_goalie,main_goalie_continuity
0,1995,hebergu01,NaN,0
26,1996,hebergu01,hebergu01,1
52,1997,hebergu01,hebergu01,1
78,1998,hebergu01,hebergu01,1
105,1999,hebergu01,hebergu01,1
133,2000,hebergu01,hebergu01,1
163,2001,gigueje01,hebergu01,0
193,2002,gigueje01,gigueje01,1
223,2003,gigueje01,gigueje01,1
253,2005,gigueje01,gigueje01,1


In [172]:
# Use only goalie continuity known before the predicted season
main_goalies["previous_main_goalie_continuity"] = (
    main_goalies
    .groupby("franchID")["main_goalie_continuity"]
    .shift(1)
)

In [173]:
main_goalies[
    main_goalies["franchID"] == "ANA"
][[
    "year",
    "main_goalie_continuity",
    "previous_main_goalie_continuity"
]]

,year,main_goalie_continuity,previous_main_goalie_continuity
0,1995,0,NaN
26,1996,1,0.0
52,1997,1,1.0
78,1998,1,1.0
105,1999,1,1.0
133,2000,1,1.0
163,2001,0,1.0
193,2002,1,0.0
223,2003,1,1.0
253,2005,1,1.0


In [174]:
# Merge previous main-goalie continuity with team data
recent_teams = recent_teams.merge(
    main_goalies[[
        "franchID",
        "year",
        "previous_main_goalie_continuity"
    ]],
    on=["franchID", "year"],
    how="left"
)

In [175]:
recent_teams[
    ["year", "tmID", "name", "previous_main_goalie_continuity"]
].head(15)

,year,tmID,name,previous_main_goalie_continuity
0,1995,ANA,Mighty Ducks of Anaheim,NaN
1,1996,ANA,Mighty Ducks of Anaheim,0.0
2,1997,ANA,Mighty Ducks of Anaheim,1.0
3,1998,ANA,Mighty Ducks of Anaheim,1.0
4,1999,ANA,Mighty Ducks of Anaheim,1.0
5,2000,ANA,Mighty Ducks of Anaheim,1.0
6,2001,ANA,Mighty Ducks of Anaheim,1.0
7,2002,ANA,Mighty Ducks of Anaheim,0.0
8,2003,ANA,Mighty Ducks of Anaheim,1.0
9,2005,ANA,Mighty Ducks of Anaheim,1.0


## SVR with Roster and Goalie Continuity

The tuned SVR model is extended with two continuity features:

- previous roster continuity
- previous main-goalie continuity

These features aim to capture changes in team composition and goalkeeper stability before the predicted season.

In [176]:
model_data_goalie_roster = recent_teams.dropna(
    subset=[
        "previous_W",
        "previous_L",
        "previous_GF",
        "previous_GA",
        "previous_Pts",
        "previous_player_G",
        "previous_player_A",
        "previous_player_Pts",
        "previous_player_SOG",
        "previous_goalie_W",
        "previous_goalie_L",
        "previous_goalie_SHO",
        "previous_goalie_GA",
        "previous_goalie_SA",
        "previous_roster_continuity",
        "previous_main_goalie_continuity"
    ]
)

model_data_goalie_roster.shape

(343, 57)

In [177]:
features_goalie_roster = [
    "previous_W",
    "previous_L",
    "previous_GF",
    "previous_GA",
    "previous_Pts",
    "previous_player_G",
    "previous_player_A",
    "previous_player_Pts",
    "previous_player_SOG",
    "previous_goalie_W",
    "previous_goalie_L",
    "previous_goalie_SHO",
    "previous_goalie_GA",
    "previous_goalie_SA",
    "previous_roster_continuity",
    "previous_main_goalie_continuity"
]

In [178]:
train_gr = model_data_goalie_roster[
    model_data_goalie_roster["year"] <= 2006
]

test_gr = model_data_goalie_roster[
    model_data_goalie_roster["year"] >= 2007
]

X_train_gr = train_gr[features_goalie_roster]
y_train_gr = train_gr["Pts"]

X_test_gr = test_gr[features_goalie_roster]
y_test_gr = test_gr["Pts"]

gr_scaler = StandardScaler()

X_train_gr_scaled = gr_scaler.fit_transform(X_train_gr)
X_test_gr_scaled = gr_scaler.transform(X_test_gr)

svr_goalie_roster = SVR(
    kernel="rbf",
    C=1,
    epsilon=0.1
)

svr_goalie_roster.fit(
    X_train_gr_scaled,
    y_train_gr
)

y_pred_gr = svr_goalie_roster.predict(
    X_test_gr_scaled
)

In [179]:
gr_rank_error, gr_rank_corr = evaluate_ranking(
    test_gr,
    y_pred_gr
)

gr_rank_error, gr_rank_corr

(np.float64(6.644444444444445), np.float64(0.5149451292274948))

### Main Goalie Continuity Evaluation

Adding previous main-goalie continuity did not improve ranking performance.

The model achieved:

- Mean Rank Error: 6.64 positions
- Rank Correlation: 0.51

Both metrics are slightly worse than the model using only roster continuity.

Therefore, `previous_main_goalie_continuity` is not retained in the final feature set.

## Final SVR Variant Comparison

Two tuned SVR variants are compared:

- the original tuned SVR using 373 observations
- the tuned SVR with roster continuity using 343 observations

Because the roster continuity feature requires additional historical information, it reduces the available sample size. Therefore, both predictive performance and dataset coverage must be considered when selecting the final model.

In [180]:
svr_variant_comparison = pd.DataFrame({
    "Model": [
        "SVR Tuned",
        "SVR Tuned + Roster Continuity"
    ],
    "Observations": [
        373,
        343
    ],
    "Mean Rank Error": [
        svr_tuned_rank_error,
        roster_rank_error
    ],
    "Rank Correlation": [
        svr_tuned_rank_corr,
        roster_rank_corr
    ]
})

svr_variant_comparison

,Model,Observations,Mean Rank Error,Rank Correlation
0,SVR Tuned,373,6.577778,0.507792
1,SVR Tuned + Roster Continuity,343,6.600000,0.522545


### Final SVR Model Selection

The tuned SVR model without roster continuity is selected as the final model.

It uses a larger dataset and achieves a slightly lower mean ranking error:

- 373 observations
- Mean Rank Error: 6.58 positions
- Rank Correlation: 0.51

Adding roster continuity slightly improves rank correlation, but reduces the available dataset and does not improve the mean ranking error.

Therefore, the tuned SVR model is selected as the final model for Question 1.

In [181]:
# Create final ranking predictions using the tuned SVR model
final_svr_ranking = test_data[
    ["year", "tmID", "name", "Pts"]
].copy()

final_svr_ranking["Predicted_Pts"] = y_pred_svr_tuned

final_svr_ranking["Actual_Rank"] = (
    final_svr_ranking
    .groupby("year")["Pts"]
    .rank(method="min", ascending=False)
    .astype(int)
)

final_svr_ranking["Predicted_Rank"] = (
    final_svr_ranking
    .groupby("year")["Predicted_Pts"]
    .rank(method="min", ascending=False)
    .astype(int)
)

In [182]:
final_svr_ranking[
    final_svr_ranking["year"] == 2009
][[
    "Predicted_Rank",
    "name",
    "Predicted_Pts",
    "Actual_Rank",
    "Pts"
]].sort_values("Predicted_Rank")

,Predicted_Rank,name,Predicted_Pts,Actual_Rank,Pts
92,1,Chicago Blackhawks,96.995450,3,112
322,2,San Jose Sharks,96.923901,2,113
224,3,New Jersey Devils,96.292624,5,103
27,4,Boston Bruins,95.955856,14,91
378,5,Vancouver Canucks,93.765111,5,103
69,6,Carolina Hurricanes,93.504898,24,80
392,7,Washington Capitals,92.886260,1,121
308,8,Pittsburgh Penguins,92.570941,8,101
134,9,Detroit Red Wings,91.659977,7,102
55,10,Calgary Flames,91.654520,15,90


## Team vs Team Data Exploration

The `TeamVsTeam` dataset contains team performance against individual opponents.

This dataset may provide additional information about team strength and consistency that is not captured by overall season totals.

In [183]:
# Load the TeamVsTeam dataset
team_vs_team_df = pd.read_csv("../hockey_development_data/TeamVsTeam.csv")

# Clean column names and identifiers
team_vs_team_df.columns = team_vs_team_df.columns.str.strip()
team_vs_team_df["tmID"] = team_vs_team_df["tmID"].str.strip()
team_vs_team_df["oppID"] = team_vs_team_df["oppID"].str.strip()

In [184]:
team_vs_team_df.head()

,year,lgID,tmID,oppID,W,L,T,OTL
0,1909,NHA,COB,HAI,1,1,0.0,
1,1909,NHA,COB,LES,2,0,0.0,
2,1909,NHA,COB,MOS,1,1,0.0,
3,1909,NHA,COB,MOW,0,2,0.0,
4,1909,NHA,COB,OT1,0,2,0.0,


In [185]:
team_vs_team_df.shape

(23862, 8)

In [186]:
team_vs_team_df[
    team_vs_team_df["year"] >= 1995
].shape

(10786, 8)

In [187]:
recent_tvt = team_vs_team_df[
    team_vs_team_df["year"] >= 1995
].copy()

recent_tvt[
    ["W", "L", "T", "OTL"]
].dtypes

W      int64
L      int64
T        str
OTL      str
dtype: object

In [188]:
recent_tvt[
    ["W", "L", "T", "OTL"]
].head(10)

,W,L,T,OTL
13076,1,1,0.0,
13077,2,0,0.0,
13078,3,2,0.0,
13079,0,3,1.0,
13080,3,3,0.0,
13081,1,3,0.0,
13082,0,3,1.0,
13083,1,4,0.0,
13084,0,1,1.0,
13085,0,2,0.0,


In [189]:
# Convert tie and overtime-loss columns to numeric
for col in ["T", "OTL"]:
    recent_tvt[col] = pd.to_numeric(
        recent_tvt[col].str.strip(),
        errors="coerce"
    )

In [190]:
recent_tvt[["W", "L", "T", "OTL"]].dtypes

W        int64
L        int64
T      float64
OTL    float64
dtype: object

In [191]:
# Calculate games played against each opponent
recent_tvt["games_vs_opponent"] = (
    recent_tvt["W"]
    + recent_tvt["L"]
    + recent_tvt["T"].fillna(0)
    + recent_tvt["OTL"].fillna(0)
)

# Calculate win rate against each opponent
recent_tvt["win_rate_vs_opponent"] = (
    recent_tvt["W"] / recent_tvt["games_vs_opponent"]
)

In [192]:
# Aggregate matchup performance at team-season level
team_matchup_features = (
    recent_tvt
    .groupby(["year", "tmID"])
    .agg(
        avg_win_rate_vs_opponent=("win_rate_vs_opponent", "mean"),
        std_win_rate_vs_opponent=("win_rate_vs_opponent", "std")
    )
    .reset_index()
)

team_matchup_features.head()

,year,tmID,avg_win_rate_vs_opponent,std_win_rate_vs_opponent
0,1995,ANA,0.443333,0.358527
1,1995,BOS,0.463333,0.323214
2,1995,BUF,0.381333,0.273325
3,1995,CAL,0.418667,0.328976
4,1995,CHI,0.518667,0.308833


In [193]:
# Add franchise identifiers
team_matchup_features = team_matchup_features.merge(
    recent_teams[
        ["year", "tmID", "franchID"]
    ].drop_duplicates(),
    on=["year", "tmID"],
    how="left"
)

In [194]:
# Sort by franchise and season
team_matchup_features = team_matchup_features.sort_values(
    ["franchID", "year"]
)

# Create previous-season matchup features
team_matchup_features["previous_avg_win_rate_vs_opponent"] = (
    team_matchup_features
    .groupby("franchID")["avg_win_rate_vs_opponent"]
    .shift(1)
)

team_matchup_features["previous_std_win_rate_vs_opponent"] = (
    team_matchup_features
    .groupby("franchID")["std_win_rate_vs_opponent"]
    .shift(1)
)

In [195]:
team_matchup_features[
    team_matchup_features["franchID"] == "ANA"
][[
    "year",
    "avg_win_rate_vs_opponent",
    "previous_avg_win_rate_vs_opponent",
    "std_win_rate_vs_opponent",
    "previous_std_win_rate_vs_opponent"
]]

,year,avg_win_rate_vs_opponent,previous_avg_win_rate_vs_opponent,std_win_rate_vs_opponent,previous_std_win_rate_vs_opponent
0,1995,0.443333,NaN,0.358527,NaN
26,1996,0.450000,0.443333,0.322677,0.358527
52,1997,0.332667,0.450000,0.261029,0.322677
78,1998,0.390385,0.332667,0.298819,0.261029
105,1999,0.386420,0.390385,0.373137,0.298819
133,2000,0.362069,0.386420,0.380708,0.373137
163,2001,0.339655,0.362069,0.360154,0.380708
193,2002,0.460345,0.339655,0.372583,0.360154
223,2003,0.373563,0.460345,0.369752,0.372583
253,2005,0.510417,0.373563,0.395141,0.369752


In [196]:
# Merge previous matchup features with team data
recent_teams = recent_teams.merge(
    team_matchup_features[[
        "franchID",
        "year",
        "previous_avg_win_rate_vs_opponent",
        "previous_std_win_rate_vs_opponent"
    ]],
    on=["franchID", "year"],
    how="left"
)

In [197]:
recent_teams[
    [
        "year",
        "tmID",
        "name",
        "previous_avg_win_rate_vs_opponent",
        "previous_std_win_rate_vs_opponent"
    ]
].head(15)

,year,tmID,name,previous_avg_win_rate_vs_opponent,previous_std_win_rate_vs_opponent
0,1995,ANA,Mighty Ducks of Anaheim,NaN,NaN
1,1996,ANA,Mighty Ducks of Anaheim,0.443333,0.358527
2,1997,ANA,Mighty Ducks of Anaheim,0.450000,0.322677
3,1998,ANA,Mighty Ducks of Anaheim,0.332667,0.261029
4,1999,ANA,Mighty Ducks of Anaheim,0.390385,0.298819
5,2000,ANA,Mighty Ducks of Anaheim,0.386420,0.373137
6,2001,ANA,Mighty Ducks of Anaheim,0.362069,0.380708
7,2002,ANA,Mighty Ducks of Anaheim,0.339655,0.360154
8,2003,ANA,Mighty Ducks of Anaheim,0.460345,0.372583
9,2005,ANA,Mighty Ducks of Anaheim,0.373563,0.369752


## SVR with Team-vs-Team Features

The tuned SVR model is extended with previous-season matchup features derived from `TeamVsTeam`.

These features describe the team's average win rate against opponents and the consistency of that performance across different opponents.

In [198]:
model_data_tvt = recent_teams.dropna(
    subset=[
        "previous_W",
        "previous_L",
        "previous_GF",
        "previous_GA",
        "previous_Pts",
        "previous_player_G",
        "previous_player_A",
        "previous_player_Pts",
        "previous_player_SOG",
        "previous_goalie_W",
        "previous_goalie_L",
        "previous_goalie_SHO",
        "previous_goalie_GA",
        "previous_goalie_SA",
        "previous_avg_win_rate_vs_opponent",
        "previous_std_win_rate_vs_opponent"
    ]
)

model_data_tvt.shape

(373, 59)

In [199]:
features_tvt = [
    "previous_W",
    "previous_L",
    "previous_GF",
    "previous_GA",
    "previous_Pts",
    "previous_player_G",
    "previous_player_A",
    "previous_player_Pts",
    "previous_player_SOG",
    "previous_goalie_W",
    "previous_goalie_L",
    "previous_goalie_SHO",
    "previous_goalie_GA",
    "previous_goalie_SA",
    "previous_avg_win_rate_vs_opponent",
    "previous_std_win_rate_vs_opponent"
]

In [200]:
train_tvt = model_data_tvt[
    model_data_tvt["year"] <= 2006
]

test_tvt = model_data_tvt[
    model_data_tvt["year"] >= 2007
]

X_train_tvt = train_tvt[features_tvt]
y_train_tvt = train_tvt["Pts"]

X_test_tvt = test_tvt[features_tvt]
y_test_tvt = test_tvt["Pts"]

tvt_scaler = StandardScaler()

X_train_tvt_scaled = tvt_scaler.fit_transform(X_train_tvt)
X_test_tvt_scaled = tvt_scaler.transform(X_test_tvt)

svr_tvt_model = SVR(
    kernel="rbf",
    C=1,
    epsilon=0.1
)

svr_tvt_model.fit(
    X_train_tvt_scaled,
    y_train_tvt
)

y_pred_tvt = svr_tvt_model.predict(
    X_test_tvt_scaled
)

In [201]:
tvt_rank_error, tvt_rank_corr = evaluate_ranking(
    test_tvt,
    y_pred_tvt
)

tvt_rank_error, tvt_rank_corr

(np.float64(6.4), np.float64(0.527313945817245))

### Team-vs-Team Feature Evaluation

Adding previous-season matchup features improved ranking performance.

The tuned SVR model with `TeamVsTeam` features achieved:

- Mean Rank Error: 6.40 positions
- Rank Correlation: 0.53

This improves the previous tuned SVR model, which achieved a mean rank error of 6.58 positions and a rank correlation of 0.51.

Therefore, the `TeamVsTeam` features provide useful additional information and are retained in the current best model.

### Team-vs-Team Feature Simplification

The contribution of the matchup consistency feature is evaluated by comparing:

- previous average win rate against opponents
- previous average win rate plus its standard deviation

This helps determine whether both matchup features are necessary.

In [202]:
features_tvt_avg_only = [
    "previous_W",
    "previous_L",
    "previous_GF",
    "previous_GA",
    "previous_Pts",
    "previous_player_G",
    "previous_player_A",
    "previous_player_Pts",
    "previous_player_SOG",
    "previous_goalie_W",
    "previous_goalie_L",
    "previous_goalie_SHO",
    "previous_goalie_GA",
    "previous_goalie_SA",
    "previous_avg_win_rate_vs_opponent"
]

In [203]:
X_train_avg = train_tvt[features_tvt_avg_only]
X_test_avg = test_tvt[features_tvt_avg_only]

avg_scaler = StandardScaler()

X_train_avg_scaled = avg_scaler.fit_transform(X_train_avg)
X_test_avg_scaled = avg_scaler.transform(X_test_avg)

svr_tvt_avg_model = SVR(
    kernel="rbf",
    C=1,
    epsilon=0.1
)

svr_tvt_avg_model.fit(
    X_train_avg_scaled,
    y_train_tvt
)

y_pred_tvt_avg = svr_tvt_avg_model.predict(
    X_test_avg_scaled
)

In [204]:
avg_rank_error, avg_rank_corr = evaluate_ranking(
    test_tvt,
    y_pred_tvt_avg
)

avg_rank_error, avg_rank_corr

(np.float64(6.444444444444445), np.float64(0.5155412167739888))

### Team-vs-Team Feature Selection

Using both matchup features produced better ranking performance than using only the average win rate.

With both features:

- Mean Rank Error: 6.40 positions
- Rank Correlation: 0.53

Using only the average win rate resulted in slightly worse performance.

Therefore, both `previous_avg_win_rate_vs_opponent` and `previous_std_win_rate_vs_opponent` are retained.

## Team Splits Data Exploration

The `TeamSplits` dataset contains team performance split by different game situations or contexts.

The goal is to identify whether these splits provide additional historical information that can improve next-season ranking prediction.

In [205]:
# Load the TeamSplits dataset
team_splits_df = pd.read_csv("../hockey_development_data/TeamSplits.csv")

# Clean column names
team_splits_df.columns = team_splits_df.columns.str.strip()

In [206]:
team_splits_df.head()

,year,lgID,tmID,hW,hL,hT,hOTL,rW,rL,rT,...,FebT,FebOL,MarW,MarL,MarT,MarOL,AprW,AprL,AprT,AprOL
0,1909,NHA,COB,2,4,0.0,,2,4,0.0,...,0.0,,1.0,4.0,0.0,,,,,
1,1909,NHA,HAI,3,3,0.0,,1,5,0.0,...,0.0,,1.0,0.0,0.0,,,,,
2,1909,NHA,LES,2,4,0.0,,0,6,0.0,...,0.0,,1.0,2.0,0.0,,,,,
3,1909,NHA,MOS,3,2,1.0,,0,6,0.0,...,0.0,,1.0,1.0,0.0,,,,,
4,1909,NHA,MOW,6,0,0.0,,5,1,0.0,...,0.0,,3.0,0.0,0.0,,,,,


In [207]:
team_splits_df.columns

Index(['year', 'lgID', 'tmID', 'hW', 'hL', 'hT', 'hOTL', 'rW', 'rL', 'rT',
       'rOTL', 'SepW', 'SepL', 'SepT', 'SepOL', 'OctW', 'OctL', 'OctT',
       'OctOL', 'NovW', 'NovL', 'NovT', 'NovOL', 'DecW', 'DecL', 'DecT',
       'DecOL', 'JanW', 'JanL', 'JanT', 'JanOL', 'FebW', 'FebL', 'FebT',
       'FebOL', 'MarW', 'MarL', 'MarT', 'MarOL', 'AprW', 'AprL', 'AprT',
       'AprOL'],
      dtype='str')

In [208]:
team_splits_df.shape

(1459, 43)

In [209]:
team_splits_df[
    team_splits_df["year"] >= 1995
].shape

(403, 43)

### Home and Road Performance

The `TeamSplits` dataset separates team performance between home and road games.

Previous-season home and road win rates may provide additional information about team strength and consistency beyond overall season totals.

In [211]:
# Restrict to the recent modelling period
recent_splits = team_splits_df[
    team_splits_df["year"] >= 1995
].copy()

# Clean team identifiers
recent_splits["tmID"] = recent_splits["tmID"].str.strip()

In [212]:
recent_splits[
    ["hW", "hL", "hT", "hOTL", "rW", "rL", "rT", "rOTL"]
].dtypes

hW      int64
hL      int64
hT        str
hOTL      str
rW      int64
rL      int64
rT        str
rOTL      str
dtype: object

In [213]:
# Convert home/road tie and overtime-loss columns to numeric
for col in ["hT", "hOTL", "rT", "rOTL"]:
    recent_splits[col] = pd.to_numeric(
        recent_splits[col].str.strip(),
        errors="coerce"
    )

In [214]:
# Calculate home and road games
recent_splits["home_games"] = (
    recent_splits["hW"]
    + recent_splits["hL"]
    + recent_splits["hT"].fillna(0)
    + recent_splits["hOTL"].fillna(0)
)

recent_splits["road_games"] = (
    recent_splits["rW"]
    + recent_splits["rL"]
    + recent_splits["rT"].fillna(0)
    + recent_splits["rOTL"].fillna(0)
)

# Calculate home and road win rates
recent_splits["home_win_rate"] = (
    recent_splits["hW"] / recent_splits["home_games"]
)

recent_splits["road_win_rate"] = (
    recent_splits["rW"] / recent_splits["road_games"]
)

In [215]:
recent_splits[
    ["year", "tmID", "home_win_rate", "road_win_rate"]
].head()

,year,tmID,home_win_rate,road_win_rate
1056,1995,ANA,0.536585,0.317073
1057,1995,BOS,0.536585,0.439024
1058,1995,BUF,0.463415,0.341463
1059,1995,CAL,0.439024,0.390244
1060,1995,CHI,0.536585,0.439024


In [216]:
# Add franchise identifiers
recent_splits = recent_splits.merge(
    recent_teams[
        ["year", "tmID", "franchID"]
    ].drop_duplicates(),
    on=["year", "tmID"],
    how="left"
)

In [217]:
# Sort by franchise and season
recent_splits = recent_splits.sort_values(
    ["franchID", "year"]
)

# Create previous-season home and road win-rate features
recent_splits["previous_home_win_rate"] = (
    recent_splits
    .groupby("franchID")["home_win_rate"]
    .shift(1)
)

recent_splits["previous_road_win_rate"] = (
    recent_splits
    .groupby("franchID")["road_win_rate"]
    .shift(1)
)

In [218]:
recent_splits[
    recent_splits["franchID"] == "ANA"
][[
    "year",
    "home_win_rate",
    "previous_home_win_rate",
    "road_win_rate",
    "previous_road_win_rate"
]]

,year,home_win_rate,previous_home_win_rate,road_win_rate,previous_road_win_rate
0,1995,0.536585,NaN,0.317073,NaN
26,1996,0.560976,0.536585,0.317073,0.317073
52,1997,0.292683,0.560976,0.341463,0.317073
78,1998,0.512195,0.292683,0.341463,0.341463
105,1999,0.463415,0.512195,0.365854,0.341463
133,2000,0.365854,0.463415,0.243902,0.365854
163,2001,0.365854,0.365854,0.341463,0.243902
193,2002,0.536585,0.365854,0.439024,0.341463
223,2003,0.463415,0.536585,0.243902,0.439024
253,2005,0.634146,0.463415,0.414634,0.243902


## SVR with Team-vs-Team and Team Splits

The current best SVR model is extended with previous-season home and road win rates.

These features aim to capture whether a team performs differently at home and away, adding information beyond overall season and opponent-level performance.

In [219]:
# Merge previous home and road performance with team data
recent_teams = recent_teams.merge(
    recent_splits[[
        "franchID",
        "year",
        "previous_home_win_rate",
        "previous_road_win_rate"
    ]],
    on=["franchID", "year"],
    how="left"
)

In [220]:
model_data_tvt_splits = recent_teams.dropna(
    subset=[
        "previous_W",
        "previous_L",
        "previous_GF",
        "previous_GA",
        "previous_Pts",
        "previous_player_G",
        "previous_player_A",
        "previous_player_Pts",
        "previous_player_SOG",
        "previous_goalie_W",
        "previous_goalie_L",
        "previous_goalie_SHO",
        "previous_goalie_GA",
        "previous_goalie_SA",
        "previous_avg_win_rate_vs_opponent",
        "previous_std_win_rate_vs_opponent",
        "previous_home_win_rate",
        "previous_road_win_rate"
    ]
)

model_data_tvt_splits.shape

(373, 61)

In [221]:
features_tvt_splits = [
    "previous_W",
    "previous_L",
    "previous_GF",
    "previous_GA",
    "previous_Pts",
    "previous_player_G",
    "previous_player_A",
    "previous_player_Pts",
    "previous_player_SOG",
    "previous_goalie_W",
    "previous_goalie_L",
    "previous_goalie_SHO",
    "previous_goalie_GA",
    "previous_goalie_SA",
    "previous_avg_win_rate_vs_opponent",
    "previous_std_win_rate_vs_opponent",
    "previous_home_win_rate",
    "previous_road_win_rate"
]

In [222]:
train_tvt_splits = model_data_tvt_splits[
    model_data_tvt_splits["year"] <= 2006
]

test_tvt_splits = model_data_tvt_splits[
    model_data_tvt_splits["year"] >= 2007
]

X_train_tvt_splits = train_tvt_splits[features_tvt_splits]
y_train_tvt_splits = train_tvt_splits["Pts"]

X_test_tvt_splits = test_tvt_splits[features_tvt_splits]
y_test_tvt_splits = test_tvt_splits["Pts"]

splits_scaler = StandardScaler()

X_train_tvt_splits_scaled = splits_scaler.fit_transform(
    X_train_tvt_splits
)

X_test_tvt_splits_scaled = splits_scaler.transform(
    X_test_tvt_splits
)

In [223]:
svr_tvt_splits_model = SVR(
    kernel="rbf",
    C=1,
    epsilon=0.1
)

svr_tvt_splits_model.fit(
    X_train_tvt_splits_scaled,
    y_train_tvt_splits
)

y_pred_tvt_splits = svr_tvt_splits_model.predict(
    X_test_tvt_splits_scaled
)

In [224]:
splits_rank_error, splits_rank_corr = evaluate_ranking(
    test_tvt_splits,
    y_pred_tvt_splits
)

splits_rank_error, splits_rank_corr

(np.float64(6.4222222222222225), np.float64(0.5286551427968564))

## Extended SVR Hyperparameter Tuning

The SVR model is further tuned by testing different kernels and hyperparameter combinations.

For the RBF kernel, the `gamma` parameter is also evaluated, since it controls the influence of individual training examples.

Model selection is based primarily on ranking performance using a chronological validation set.

In [225]:
features_best = [
    "previous_W",
    "previous_L",
    "previous_GF",
    "previous_GA",
    "previous_Pts",
    "previous_player_G",
    "previous_player_A",
    "previous_player_Pts",
    "previous_player_SOG",
    "previous_goalie_W",
    "previous_goalie_L",
    "previous_goalie_SHO",
    "previous_goalie_GA",
    "previous_goalie_SA",
    "previous_avg_win_rate_vs_opponent",
    "previous_std_win_rate_vs_opponent"
]

In [226]:
train_full = model_data_tvt[
    model_data_tvt["year"] <= 2006
]

inner_train = train_full[
    train_full["year"] <= 2003
]

validation = train_full[
    train_full["year"] >= 2005
]

X_inner = inner_train[features_best]
y_inner = inner_train["Pts"]

X_val = validation[features_best]
y_val = validation["Pts"]

In [227]:
svr_search_scaler = StandardScaler()

X_inner_scaled = svr_search_scaler.fit_transform(X_inner)
X_val_scaled = svr_search_scaler.transform(X_val)

In [228]:
C_values = [0.1, 1, 10, 100]
epsilon_values = [0.1, 0.5, 1.0]
gamma_values = ["scale", 0.001, 0.01, 0.1, 1]

svr_search_results = []

for C in C_values:
    for epsilon in epsilon_values:
        for gamma in gamma_values:

            model = SVR(
                kernel="rbf",
                C=C,
                epsilon=epsilon,
                gamma=gamma
            )

            model.fit(X_inner_scaled, y_inner)

            pred = model.predict(X_val_scaled)

            rank_error, rank_corr = evaluate_ranking(
                validation,
                pred
            )

            svr_search_results.append({
                "C": C,
                "epsilon": epsilon,
                "gamma": gamma,
                "Rank_Error": rank_error,
                "Rank_Correlation": rank_corr
            })

svr_search_df = pd.DataFrame(svr_search_results)

svr_search_df.sort_values(
    ["Rank_Error", "Rank_Correlation"],
    ascending=[True, False]
).head(15)

,C,epsilon,gamma,Rank_Error,Rank_Correlation
31,10.0,0.1,0.001,6.116667,0.568617
2,0.1,0.1,0.01,6.116667,0.564454
37,10.0,0.5,0.01,6.116667,0.541451
27,1.0,1.0,0.01,6.150000,0.569274
22,1.0,0.5,0.01,6.150000,0.566864
36,10.0,0.5,0.001,6.150000,0.563797
12,0.1,1.0,0.01,6.150000,0.562921
11,0.1,1.0,0.001,6.150000,0.562702
16,1.0,0.1,0.001,6.150000,0.562702
21,1.0,0.5,0.001,6.150000,0.562702


### Extended SVR Tuning Result

Including the `gamma` hyperparameter improved validation ranking performance.

The best configuration was:

- `C = 10`
- `epsilon = 0.1`
- `gamma = 0.001`
- RBF kernel

This configuration achieved:

- Mean Rank Error: 6.12 positions
- Rank Correlation: 0.57

Therefore, this configuration is selected for final evaluation on the test set.

In [229]:
# Prepare full training and test sets
train_final = model_data_tvt[
    model_data_tvt["year"] <= 2006
]

test_final = model_data_tvt[
    model_data_tvt["year"] >= 2007
]

X_train_final = train_final[features_best]
y_train_final = train_final["Pts"]

X_test_final = test_final[features_best]
y_test_final = test_final["Pts"]

In [230]:
# Scale features
final_scaler = StandardScaler()

X_train_final_scaled = final_scaler.fit_transform(
    X_train_final
)

X_test_final_scaled = final_scaler.transform(
    X_test_final
)

In [231]:
# Train the final tuned SVR
svr_extended_model = SVR(
    kernel="rbf",
    C=10,
    epsilon=0.1,
    gamma=0.001
)

svr_extended_model.fit(
    X_train_final_scaled,
    y_train_final
)

y_pred_svr_extended = svr_extended_model.predict(
    X_test_final_scaled
)

In [232]:
extended_rank_error, extended_rank_corr = evaluate_ranking(
    test_final,
    y_pred_svr_extended
)

extended_rank_error, extended_rank_corr

(np.float64(6.7555555555555555), np.float64(0.503619465844109))

### Extended SVR Final Evaluation

Although the extended SVR tuning improved validation performance, the selected configuration performed worse on the final test set:

- Mean Rank Error: 6.76 positions
- Rank Correlation: 0.50

This suggests that the tuned configuration overfitted the validation period and did not generalize as well to unseen seasons.

Therefore, the previous SVR model with `TeamVsTeam` features remains the best-performing model.

## Player Profile Exploration

The `Master` dataset contains player characteristics such as birth year, position, height, weight, and NHL career dates.

These variables can be combined with previous-season rosters to describe team age and experience, potentially helping to explain future improvements or declines.

In [233]:
# Load the Master dataset
master_df = pd.read_csv("../hockey_development_data/Master.csv")

# Clean column names
master_df.columns = master_df.columns.str.strip()

In [234]:
master_df[
    [
        "playerID",
        "birthYear",
        "height",
        "weight",
        "pos",
        "firstNHL",
        "lastNHL"
    ]
].head()

,playerID,birthYear,height,weight,pos,firstNHL,lastNHL
0,aaltoan01,1975.0,73.0,210.0,C,1997.0,2000.0
1,abbeybr01,1951.0,73.0,185.0,D,,
2,abbotge01,1911.0,67.0,153.0,G,1943.0,1943.0
3,abbotre01,1930.0,71.0,164.0,C,1952.0,1952.0
4,abdelju01,1987.0,73.0,195.0,L,2007.0,2009.0


In [235]:
master_df[
    ["birthYear", "height", "weight", "firstNHL", "lastNHL"]
].dtypes

birthYear    str
height       str
weight       str
firstNHL     str
lastNHL      str
dtype: object

### Player Profile Data Types

Several player profile variables are stored as text.

These variables must be converted to numeric format before creating team-level age and experience features.

In [236]:
# Convert player profile columns to numeric
master_numeric_cols = [
    "birthYear",
    "height",
    "weight",
    "firstNHL",
    "lastNHL"
]

for col in master_numeric_cols:
    master_df[col] = pd.to_numeric(
        master_df[col].str.strip(),
        errors="coerce"
    )

In [237]:
master_df[master_numeric_cols].dtypes

birthYear    float64
height       float64
weight       float64
firstNHL     float64
lastNHL      float64
dtype: object

In [238]:
# Add player profile information to scoring data
scoring_profiles = recent_scoring.merge(
    master_df[
        ["playerID", "birthYear", "firstNHL", "height", "weight"]
    ],
    on="playerID",
    how="left"
)

In [239]:
# Calculate player age and NHL experience for each season
scoring_profiles["age"] = (
    scoring_profiles["year"] - scoring_profiles["birthYear"]
)

scoring_profiles["nhl_experience"] = (
    scoring_profiles["year"] - scoring_profiles["firstNHL"]
)

In [240]:
scoring_profiles[
    ["year", "tmID", "playerID", "age", "nhl_experience"]
].head(10)

,year,tmID,playerID,age,nhl_experience
0,1997,ANA,aaltoan01,22.0,0.0
1,1998,ANA,aaltoan01,23.0,1.0
2,1999,ANA,aaltoan01,24.0,2.0
3,2000,ANA,aaltoan01,25.0,3.0
4,2007,DET,abdelju01,20.0,0.0
5,2008,DET,abdelju01,21.0,1.0
6,2009,DET,abdelju01,22.0,2.0
7,2002,PHO,abidra01,22.0,0.0
8,2002,PIT,abidra01,22.0,0.0
9,2003,PIT,abidra01,23.0,1.0


In [241]:
# Keep one record per player, team and season
team_player_profiles = (
    scoring_profiles[
        ["year", "tmID", "playerID", "age", "nhl_experience"]
    ]
    .drop_duplicates(
        subset=["year", "tmID", "playerID"]
    )
)

In [242]:
# Aggregate player profile information at team-season level
team_profile_features = (
    team_player_profiles
    .groupby(["year", "tmID"])
    .agg(
        average_age=("age", "mean"),
        average_experience=("nhl_experience", "mean")
    )
    .reset_index()
)

team_profile_features.head()

,year,tmID,average_age,average_experience
0,1995,ANA,26.047619,3.857143
1,1995,BOS,27.000000,5.128205
2,1995,BUF,25.450000,4.600000
3,1995,CAL,26.219512,4.951220
4,1995,CHI,26.696970,6.272727


In [243]:
# Add franchise identifiers
team_profile_features = team_profile_features.merge(
    recent_teams[
        ["year", "tmID", "franchID"]
    ].drop_duplicates(),
    on=["year", "tmID"],
    how="left"
)

In [244]:
# Sort by franchise and season
team_profile_features = team_profile_features.sort_values(
    ["franchID", "year"]
)

# Create previous-season profile features
team_profile_features["previous_average_age"] = (
    team_profile_features
    .groupby("franchID")["average_age"]
    .shift(1)
)

team_profile_features["previous_average_experience"] = (
    team_profile_features
    .groupby("franchID")["average_experience"]
    .shift(1)
)

In [245]:
team_profile_features[
    team_profile_features["franchID"] == "ANA"
][[
    "year",
    "average_age",
    "previous_average_age",
    "average_experience",
    "previous_average_experience"
]]

,year,average_age,previous_average_age,average_experience,previous_average_experience
0,1995,26.047619,NaN,3.857143,NaN
26,1996,26.100000,26.047619,4.500000,3.857143
52,1997,25.755556,26.100000,4.200000,4.500000
78,1998,26.214286,25.755556,4.571429,4.200000
105,1999,26.875000,26.214286,5.281250,4.571429
133,2000,26.371429,26.875000,4.485714,5.281250
163,2001,27.096774,26.371429,5.290323,4.485714
193,2002,27.575758,27.096774,5.606061,5.290323
223,2003,26.600000,27.575758,4.514286,5.606061
253,2005,28.090909,26.600000,6.909091,4.514286


In [246]:
# Merge previous player profile features with team data
recent_teams = recent_teams.merge(
    team_profile_features[[
        "franchID",
        "year",
        "previous_average_age",
        "previous_average_experience"
    ]],
    on=["franchID", "year"],
    how="left"
)

In [247]:
model_data_profiles = recent_teams.dropna(
    subset=[
        "previous_W",
        "previous_L",
        "previous_GF",
        "previous_GA",
        "previous_Pts",
        "previous_player_G",
        "previous_player_A",
        "previous_player_Pts",
        "previous_player_SOG",
        "previous_goalie_W",
        "previous_goalie_L",
        "previous_goalie_SHO",
        "previous_goalie_GA",
        "previous_goalie_SA",
        "previous_avg_win_rate_vs_opponent",
        "previous_std_win_rate_vs_opponent",
        "previous_average_age",
        "previous_average_experience"
    ]
)

model_data_profiles.shape

(373, 63)

In [248]:
features_profiles = [
    "previous_W",
    "previous_L",
    "previous_GF",
    "previous_GA",
    "previous_Pts",
    "previous_player_G",
    "previous_player_A",
    "previous_player_Pts",
    "previous_player_SOG",
    "previous_goalie_W",
    "previous_goalie_L",
    "previous_goalie_SHO",
    "previous_goalie_GA",
    "previous_goalie_SA",
    "previous_avg_win_rate_vs_opponent",
    "previous_std_win_rate_vs_opponent",
    "previous_average_age",
    "previous_average_experience"
]

In [249]:
train_profiles = model_data_profiles[
    model_data_profiles["year"] <= 2006
]

test_profiles = model_data_profiles[
    model_data_profiles["year"] >= 2007
]

X_train_profiles = train_profiles[features_profiles]
y_train_profiles = train_profiles["Pts"]

X_test_profiles = test_profiles[features_profiles]
y_test_profiles = test_profiles["Pts"]

In [250]:
profiles_scaler = StandardScaler()

X_train_profiles_scaled = profiles_scaler.fit_transform(
    X_train_profiles
)

X_test_profiles_scaled = profiles_scaler.transform(
    X_test_profiles
)

In [251]:
svr_profiles_model = SVR(
    kernel="rbf",
    C=1,
    epsilon=0.1
)

svr_profiles_model.fit(
    X_train_profiles_scaled,
    y_train_profiles
)

y_pred_profiles = svr_profiles_model.predict(
    X_test_profiles_scaled
)

In [252]:
profiles_rank_error, profiles_rank_corr = evaluate_ranking(
    test_profiles,
    y_pred_profiles
)

profiles_rank_error, profiles_rank_corr

(np.float64(6.444444444444445), np.float64(0.5228432892185398))

### Player Profile Feature Evaluation

Adding previous-season average age and NHL experience did not improve ranking performance.

The model achieved:

- Mean Rank Error: 6.44 positions
- Rank Correlation: 0.52

This is slightly worse than the model using only the current `TeamVsTeam` features.

Therefore, average age and experience are not retained in the current best feature set.

In [253]:
best_results = test_tvt[
    ["year", "tmID", "name", "Pts"]
].copy()

best_results["Predicted_Pts"] = y_pred_tvt

best_results["Actual_Rank"] = (
    best_results
    .groupby("year")["Pts"]
    .rank(method="min", ascending=False)
    .astype(int)
)

best_results["Predicted_Rank"] = (
    best_results
    .groupby("year")["Predicted_Pts"]
    .rank(method="min", ascending=False)
    .astype(int)
)

best_results["Rank_Error"] = (
    best_results["Actual_Rank"]
    - best_results["Predicted_Rank"]
).abs()

In [254]:
best_results[
    best_results["year"] == 2009
][[
    "Predicted_Rank",
    "name",
    "Predicted_Pts",
    "Actual_Rank",
    "Pts",
    "Rank_Error"
]].sort_values("Predicted_Rank")

,Predicted_Rank,name,Predicted_Pts,Actual_Rank,Pts,Rank_Error
92,1,Chicago Blackhawks,97.320040,3,112,2
322,2,San Jose Sharks,97.111886,2,113,0
224,3,New Jersey Devils,96.761714,5,103,2
27,4,Boston Bruins,95.442762,14,91,10
378,5,Vancouver Canucks,94.393410,5,103,0
69,6,Carolina Hurricanes,94.103622,24,80,18
392,7,Washington Capitals,93.613207,1,121,6
308,8,Pittsburgh Penguins,93.525563,8,101,0
55,9,Calgary Flames,93.068829,15,90,6
134,10,Detroit Red Wings,92.121975,7,102,3


## Young and Veteran Player Features

Average team age may hide differences in roster composition.

Two additional features are created to represent the proportion of young and veteran players in the previous-season roster.

In [255]:
# Identify young and veteran players
team_player_profiles["is_young"] = (
    team_player_profiles["age"] < 25
).astype(int)

team_player_profiles["is_veteran"] = (
    team_player_profiles["age"] >= 30
).astype(int)

In [ ]:
team_age_features = (
    team_player_profiles
    .groupby(["year", "tmID"])
    .agg(
        young_player_ratio=("is_young", "mean"),
        veteran_player_ratio=("is_veteran", "mean")
    )
    .reset_index()
)

team_age_features.head()

,year,tmID,young_player_ratio,veteran_player_ratio
0,1995,ANA,0.357143,0.166667
1,1995,BOS,0.282051,0.256410
2,1995,BUF,0.450000,0.200000
3,1995,CAL,0.341463,0.219512
4,1995,CHI,0.212121,0.272727


In [257]:
# Add franchise identifiers
team_age_features = team_age_features.merge(
    recent_teams[
        ["year", "tmID", "franchID"]
    ].drop_duplicates(),
    on=["year", "tmID"],
    how="left"
)

In [258]:
# Sort by franchise and season
team_age_features = team_age_features.sort_values(
    ["franchID", "year"]
)

# Create previous-season age composition features
team_age_features["previous_young_player_ratio"] = (
    team_age_features
    .groupby("franchID")["young_player_ratio"]
    .shift(1)
)

team_age_features["previous_veteran_player_ratio"] = (
    team_age_features
    .groupby("franchID")["veteran_player_ratio"]
    .shift(1)
)

In [259]:
team_age_features[
    team_age_features["franchID"] == "ANA"
][[
    "year",
    "young_player_ratio",
    "previous_young_player_ratio",
    "veteran_player_ratio",
    "previous_veteran_player_ratio"
]]

,year,young_player_ratio,previous_young_player_ratio,veteran_player_ratio,previous_veteran_player_ratio
0,1995,0.357143,NaN,0.166667,NaN
26,1996,0.350000,0.357143,0.200000,0.166667
52,1997,0.444444,0.350000,0.244444,0.200000
78,1998,0.392857,0.444444,0.142857,0.244444
105,1999,0.312500,0.392857,0.250000,0.142857
133,2000,0.371429,0.312500,0.228571,0.250000
163,2001,0.290323,0.371429,0.290323,0.228571
193,2002,0.212121,0.290323,0.272727,0.290323
223,2003,0.342857,0.212121,0.228571,0.272727
253,2005,0.181818,0.342857,0.363636,0.228571


In [260]:
# Merge previous age-composition features with team data
recent_teams = recent_teams.merge(
    team_age_features[[
        "franchID",
        "year",
        "previous_young_player_ratio",
        "previous_veteran_player_ratio"
    ]],
    on=["franchID", "year"],
    how="left"
)

In [261]:
model_data_age_ratio = recent_teams.dropna(
    subset=[
        "previous_W",
        "previous_L",
        "previous_GF",
        "previous_GA",
        "previous_Pts",
        "previous_player_G",
        "previous_player_A",
        "previous_player_Pts",
        "previous_player_SOG",
        "previous_goalie_W",
        "previous_goalie_L",
        "previous_goalie_SHO",
        "previous_goalie_GA",
        "previous_goalie_SA",
        "previous_avg_win_rate_vs_opponent",
        "previous_std_win_rate_vs_opponent",
        "previous_young_player_ratio",
        "previous_veteran_player_ratio"
    ]
)

model_data_age_ratio.shape

(373, 65)

In [262]:
features_age_ratio = [
    "previous_W",
    "previous_L",
    "previous_GF",
    "previous_GA",
    "previous_Pts",
    "previous_player_G",
    "previous_player_A",
    "previous_player_Pts",
    "previous_player_SOG",
    "previous_goalie_W",
    "previous_goalie_L",
    "previous_goalie_SHO",
    "previous_goalie_GA",
    "previous_goalie_SA",
    "previous_avg_win_rate_vs_opponent",
    "previous_std_win_rate_vs_opponent",
    "previous_young_player_ratio",
    "previous_veteran_player_ratio"
]

In [263]:
train_age_ratio = model_data_age_ratio[
    model_data_age_ratio["year"] <= 2006
]

test_age_ratio = model_data_age_ratio[
    model_data_age_ratio["year"] >= 2007
]

X_train_age_ratio = train_age_ratio[features_age_ratio]
y_train_age_ratio = train_age_ratio["Pts"]

X_test_age_ratio = test_age_ratio[features_age_ratio]
y_test_age_ratio = test_age_ratio["Pts"]

age_ratio_scaler = StandardScaler()

X_train_age_ratio_scaled = age_ratio_scaler.fit_transform(
    X_train_age_ratio
)

X_test_age_ratio_scaled = age_ratio_scaler.transform(
    X_test_age_ratio
)

In [264]:
svr_age_ratio_model = SVR(
    kernel="rbf",
    C=1,
    epsilon=0.1
)

svr_age_ratio_model.fit(
    X_train_age_ratio_scaled,
    y_train_age_ratio
)

y_pred_age_ratio = svr_age_ratio_model.predict(
    X_test_age_ratio_scaled
)

In [265]:
age_ratio_rank_error, age_ratio_rank_corr = evaluate_ranking(
    test_age_ratio,
    y_pred_age_ratio
)

age_ratio_rank_error, age_ratio_rank_corr

(np.float64(6.355555555555555), np.float64(0.5206079609191876))

### Age Composition Feature Evaluation

Adding previous-season young and veteran player ratios slightly improved the mean ranking error:

- Mean Rank Error: 6.36 positions
- Rank Correlation: 0.52

Although rank correlation decreased slightly, the average positional error improved.

Therefore, the age-composition features are retained in the current best model.

## Player Position Composition

Team roster structure may also depend on the distribution of player positions.

Position-based roster features are created to describe the proportion of forwards, defensemen and goalies in the previous-season roster.

In [266]:
# Add player position information
team_player_positions = team_player_profiles.merge(
    master_df[
        ["playerID", "pos"]
    ],
    on="playerID",
    how="left"
)

In [267]:
team_player_positions["pos"].value_counts(dropna=False)

pos
D      4622
C      3578
R      2535
L      2508
G      1285
F         8
Name: count, dtype: int64

In [268]:
# Group player positions into broader categories
team_player_positions["position_group"] = team_player_positions["pos"].replace({
    "C": "F",
    "R": "F",
    "L": "F"
})

In [269]:
# Create position indicators
team_player_positions["is_forward"] = (
    team_player_positions["position_group"] == "F"
).astype(int)

team_player_positions["is_defense"] = (
    team_player_positions["position_group"] == "D"
).astype(int)

team_player_positions["is_goalie"] = (
    team_player_positions["position_group"] == "G"
).astype(int)

In [270]:
# Aggregate position composition at team-season level
team_position_features = (
    team_player_positions
    .groupby(["year", "tmID"])
    .agg(
        forward_ratio=("is_forward", "mean"),
        defense_ratio=("is_defense", "mean"),
        goalie_ratio=("is_goalie", "mean")
    )
    .reset_index()
)

team_position_features.head()

,year,tmID,forward_ratio,defense_ratio,goalie_ratio
0,1995,ANA,0.0,0.0,0.0
1,1995,BOS,0.0,0.0,0.0
2,1995,BUF,0.0,0.0,0.0
3,1995,CAL,0.0,0.0,0.0
4,1995,CHI,0.0,0.0,0.0


In [271]:
# Clean position values
team_player_positions["pos"] = (
    team_player_positions["pos"]
    .str.strip()
)

In [272]:
team_player_positions["pos"].value_counts(dropna=False)

pos
D    4622
C    3578
R    2535
L    2508
G    1285
F       8
Name: count, dtype: int64

In [273]:
team_player_positions["position_group"] = (
    team_player_positions["pos"].replace({
        "C": "F",
        "R": "F",
        "L": "F"
    })
)

In [274]:
team_player_positions["is_forward"] = (
    team_player_positions["position_group"] == "F"
).astype(int)

team_player_positions["is_defense"] = (
    team_player_positions["position_group"] == "D"
).astype(int)

team_player_positions["is_goalie"] = (
    team_player_positions["position_group"] == "G"
).astype(int)

In [275]:
team_position_features = (
    team_player_positions
    .groupby(["year", "tmID"])
    .agg(
        forward_ratio=("is_forward", "mean"),
        defense_ratio=("is_defense", "mean"),
        goalie_ratio=("is_goalie", "mean")
    )
    .reset_index()
)

team_position_features.head()

,year,tmID,forward_ratio,defense_ratio,goalie_ratio
0,1995,ANA,0.666667,0.285714,0.047619
1,1995,BOS,0.589744,0.282051,0.128205
2,1995,BUF,0.575000,0.300000,0.125000
3,1995,CAL,0.658537,0.292683,0.048780
4,1995,CHI,0.606061,0.303030,0.090909


In [276]:
# Add franchise identifiers
team_position_features = team_position_features.merge(
    recent_teams[
        ["year", "tmID", "franchID"]
    ].drop_duplicates(),
    on=["year", "tmID"],
    how="left"
)

In [277]:
# Sort by franchise and season
team_position_features = team_position_features.sort_values(
    ["franchID", "year"]
)

# Create previous-season position features
for col in ["forward_ratio", "defense_ratio", "goalie_ratio"]:
    team_position_features[f"previous_{col}"] = (
        team_position_features
        .groupby("franchID")[col]
        .shift(1)
    )

In [278]:
team_position_features[
    team_position_features["franchID"] == "ANA"
][[
    "year",
    "forward_ratio",
    "previous_forward_ratio",
    "defense_ratio",
    "previous_defense_ratio",
    "goalie_ratio",
    "previous_goalie_ratio"
]]

,year,forward_ratio,previous_forward_ratio,defense_ratio,previous_defense_ratio,goalie_ratio,previous_goalie_ratio
0,1995,0.666667,NaN,0.285714,NaN,0.047619,NaN
26,1996,0.650000,0.666667,0.275000,0.285714,0.075000,0.047619
52,1997,0.600000,0.650000,0.333333,0.275000,0.066667,0.075000
78,1998,0.535714,0.600000,0.357143,0.333333,0.107143,0.066667
105,1999,0.656250,0.535714,0.281250,0.357143,0.062500,0.107143
133,2000,0.600000,0.656250,0.285714,0.281250,0.114286,0.062500
163,2001,0.580645,0.600000,0.322581,0.285714,0.096774,0.114286
193,2002,0.636364,0.580645,0.303030,0.322581,0.060606,0.096774
223,2003,0.600000,0.636364,0.314286,0.303030,0.085714,0.060606
253,2005,0.636364,0.600000,0.303030,0.314286,0.060606,0.085714


In [279]:
# Merge previous position-composition features with team data
recent_teams = recent_teams.merge(
    team_position_features[[
        "franchID",
        "year",
        "previous_forward_ratio",
        "previous_defense_ratio",
        "previous_goalie_ratio"
    ]],
    on=["franchID", "year"],
    how="left"
)

In [280]:
model_data_position = recent_teams.dropna(
    subset=[
        "previous_W",
        "previous_L",
        "previous_GF",
        "previous_GA",
        "previous_Pts",
        "previous_player_G",
        "previous_player_A",
        "previous_player_Pts",
        "previous_player_SOG",
        "previous_goalie_W",
        "previous_goalie_L",
        "previous_goalie_SHO",
        "previous_goalie_GA",
        "previous_goalie_SA",
        "previous_avg_win_rate_vs_opponent",
        "previous_std_win_rate_vs_opponent",
        "previous_young_player_ratio",
        "previous_veteran_player_ratio",
        "previous_forward_ratio",
        "previous_defense_ratio",
        "previous_goalie_ratio"
    ]
)

model_data_position.shape

(373, 68)

In [281]:
features_position = [
    "previous_W",
    "previous_L",
    "previous_GF",
    "previous_GA",
    "previous_Pts",
    "previous_player_G",
    "previous_player_A",
    "previous_player_Pts",
    "previous_player_SOG",
    "previous_goalie_W",
    "previous_goalie_L",
    "previous_goalie_SHO",
    "previous_goalie_GA",
    "previous_goalie_SA",
    "previous_avg_win_rate_vs_opponent",
    "previous_std_win_rate_vs_opponent",
    "previous_young_player_ratio",
    "previous_veteran_player_ratio",
    "previous_forward_ratio",
    "previous_defense_ratio",
    "previous_goalie_ratio"
]

In [282]:
train_position = model_data_position[
    model_data_position["year"] <= 2006
]

test_position = model_data_position[
    model_data_position["year"] >= 2007
]

X_train_position = train_position[features_position]
y_train_position = train_position["Pts"]

X_test_position = test_position[features_position]
y_test_position = test_position["Pts"]

position_scaler = StandardScaler()

X_train_position_scaled = position_scaler.fit_transform(
    X_train_position
)

X_test_position_scaled = position_scaler.transform(
    X_test_position
)

svr_position_model = SVR(
    kernel="rbf",
    C=1,
    epsilon=0.1
)

svr_position_model.fit(
    X_train_position_scaled,
    y_train_position
)

y_pred_position = svr_position_model.predict(
    X_test_position_scaled
)

In [283]:
position_rank_error, position_rank_corr = evaluate_ranking(
    test_position,
    y_pred_position
)

position_rank_error, position_rank_corr

(np.float64(6.288888888888889), np.float64(0.5304434054363383))

### Position Composition Evaluation

Adding previous-season roster composition by player position improved ranking performance.

The model achieved:

- Mean Rank Error: 6.29 positions
- Rank Correlation: 0.53

Both metrics improve compared with the previous best model.

Therefore, the position-composition features are retained in the current best feature set.

## Physical Player Profile Features

The `Master` dataset also contains player height and weight.

Previous-season average height and weight are tested to determine whether physical roster characteristics provide additional predictive information.

In [284]:
# Keep one record per player, team and season
team_physical_profiles = (
    scoring_profiles[
        ["year", "tmID", "playerID", "height", "weight"]
    ]
    .drop_duplicates(
        subset=["year", "tmID", "playerID"]
    )
)

In [285]:
# Aggregate physical player characteristics at team-season level
team_physical_features = (
    team_physical_profiles
    .groupby(["year", "tmID"])
    .agg(
        average_height=("height", "mean"),
        average_weight=("weight", "mean")
    )
    .reset_index()
)

team_physical_features.head()

,year,tmID,average_height,average_weight
0,1995,ANA,72.809524,200.857143
1,1995,BOS,72.461538,200.846154
2,1995,BUF,72.375000,198.075000
3,1995,CAL,72.341463,198.878049
4,1995,CHI,73.181818,201.757576


In [286]:
# Add franchise identifiers
team_physical_features = team_physical_features.merge(
    recent_teams[
        ["year", "tmID", "franchID"]
    ].drop_duplicates(),
    on=["year", "tmID"],
    how="left"
)

In [287]:
# Sort by franchise and season
team_physical_features = team_physical_features.sort_values(
    ["franchID", "year"]
)

# Create previous-season physical profile features
team_physical_features["previous_average_height"] = (
    team_physical_features
    .groupby("franchID")["average_height"]
    .shift(1)
)

team_physical_features["previous_average_weight"] = (
    team_physical_features
    .groupby("franchID")["average_weight"]
    .shift(1)
)

In [288]:
team_physical_features[
    team_physical_features["franchID"] == "ANA"
][[
    "year",
    "average_height",
    "previous_average_height",
    "average_weight",
    "previous_average_weight"
]]

,year,average_height,previous_average_height,average_weight,previous_average_weight
0,1995,72.809524,NaN,200.857143,NaN
26,1996,72.900000,72.809524,200.700000,200.857143
52,1997,73.066667,72.900000,201.822222,200.700000
78,1998,73.178571,73.066667,202.464286,201.822222
105,1999,72.750000,73.178571,202.375000,202.464286
133,2000,72.885714,72.750000,202.085714,202.375000
163,2001,73.096774,72.885714,203.064516,202.085714
193,2002,73.212121,73.096774,202.575758,203.064516
223,2003,73.342857,73.212121,204.885714,202.575758
253,2005,73.454545,73.342857,207.666667,204.885714


In [289]:
# Merge previous physical profile features with team data
recent_teams = recent_teams.merge(
    team_physical_features[[
        "franchID",
        "year",
        "previous_average_height",
        "previous_average_weight"
    ]],
    on=["franchID", "year"],
    how="left"
)

In [290]:
model_data_physical = recent_teams.dropna(
    subset=[
        "previous_W",
        "previous_L",
        "previous_GF",
        "previous_GA",
        "previous_Pts",
        "previous_player_G",
        "previous_player_A",
        "previous_player_Pts",
        "previous_player_SOG",
        "previous_goalie_W",
        "previous_goalie_L",
        "previous_goalie_SHO",
        "previous_goalie_GA",
        "previous_goalie_SA",
        "previous_avg_win_rate_vs_opponent",
        "previous_std_win_rate_vs_opponent",
        "previous_young_player_ratio",
        "previous_veteran_player_ratio",
        "previous_forward_ratio",
        "previous_defense_ratio",
        "previous_goalie_ratio",
        "previous_average_height",
        "previous_average_weight"
    ]
)

model_data_physical.shape

(373, 70)

In [291]:
features_physical = [
    "previous_W",
    "previous_L",
    "previous_GF",
    "previous_GA",
    "previous_Pts",
    "previous_player_G",
    "previous_player_A",
    "previous_player_Pts",
    "previous_player_SOG",
    "previous_goalie_W",
    "previous_goalie_L",
    "previous_goalie_SHO",
    "previous_goalie_GA",
    "previous_goalie_SA",
    "previous_avg_win_rate_vs_opponent",
    "previous_std_win_rate_vs_opponent",
    "previous_young_player_ratio",
    "previous_veteran_player_ratio",
    "previous_forward_ratio",
    "previous_defense_ratio",
    "previous_goalie_ratio",
    "previous_average_height",
    "previous_average_weight"
]

In [292]:
train_physical = model_data_physical[
    model_data_physical["year"] <= 2006
]

test_physical = model_data_physical[
    model_data_physical["year"] >= 2007
]

X_train_physical = train_physical[features_physical]
y_train_physical = train_physical["Pts"]

X_test_physical = test_physical[features_physical]
y_test_physical = test_physical["Pts"]

physical_scaler = StandardScaler()

X_train_physical_scaled = physical_scaler.fit_transform(
    X_train_physical
)

X_test_physical_scaled = physical_scaler.transform(
    X_test_physical
)

In [293]:
svr_physical_model = SVR(
    kernel="rbf",
    C=1,
    epsilon=0.1
)

svr_physical_model.fit(
    X_train_physical_scaled,
    y_train_physical
)

y_pred_physical = svr_physical_model.predict(
    X_test_physical_scaled
)

In [295]:
physical_rank_error, physical_rank_corr = evaluate_ranking(
    test_physical,
    y_pred_physical
)

physical_rank_error, physical_rank_corr

(np.float64(6.4), np.float64(0.5246315518580219))

### Physical Profile Feature Evaluation

Adding previous-season average height and weight did not improve ranking performance.

The model achieved:

- Mean Rank Error: 6.40 positions
- Rank Correlation: 0.52

Both metrics are worse than the previous best model.

Therefore, average height and weight are not retained in the final feature set.

In [303]:
# Convert March and April columns to numeric
for col in ["MarW", "MarL", "MarT", "MarOL",
            "AprW", "AprL", "AprT", "AprOL"]:
    
    recent_splits[col] = pd.to_numeric(
        recent_splits[col],
        errors="coerce"
    )

In [304]:
recent_splits[
    ["MarW", "MarL", "MarT", "MarOL",
     "AprW", "AprL", "AprT", "AprOL"]
].dtypes

MarW     float64
MarL     float64
MarT     float64
MarOL    float64
AprW     float64
AprL     float64
AprT     float64
AprOL    float64
dtype: object

In [305]:
# Calculate late-season games
recent_splits["late_games"] = (
    recent_splits["MarW"]
    + recent_splits["MarL"]
    + recent_splits["MarT"].fillna(0)
    + recent_splits["MarOL"].fillna(0)
    + recent_splits["AprW"]
    + recent_splits["AprL"]
    + recent_splits["AprT"].fillna(0)
    + recent_splits["AprOL"].fillna(0)
)

# Calculate late-season win rate
recent_splits["late_season_win_rate"] = (
    recent_splits["MarW"] + recent_splits["AprW"]
) / recent_splits["late_games"]

In [306]:
recent_splits[
    ["year", "tmID", "late_season_win_rate"]
].head()

,year,tmID,late_season_win_rate
0,1995,ANA,0.631579
26,1996,ANA,0.526316
52,1997,ANA,0.250000
78,1998,ANA,0.380952
105,1999,ANA,0.388889


In [307]:
# Sort by franchise and season
recent_splits = recent_splits.sort_values(
    ["franchID", "year"]
)

# Create previous-season late form feature
recent_splits["previous_late_season_win_rate"] = (
    recent_splits
    .groupby("franchID")["late_season_win_rate"]
    .shift(1)
)

In [308]:
recent_splits[
    recent_splits["franchID"] == "ANA"
][[
    "year",
    "late_season_win_rate",
    "previous_late_season_win_rate"
]]

,year,late_season_win_rate,previous_late_season_win_rate
0,1995,0.631579,NaN
26,1996,0.526316,0.631579
52,1997,0.250000,0.526316
78,1998,0.380952,0.250000
105,1999,0.388889,0.380952
133,2000,0.411765,0.388889
163,2001,0.400000,0.411765
193,2002,0.588235,0.400000
223,2003,0.375000,0.588235
253,2005,0.640000,0.375000


In [309]:
# Merge previous late-season form with team data
recent_teams = recent_teams.merge(
    recent_splits[[
        "franchID",
        "year",
        "previous_late_season_win_rate"
    ]],
    on=["franchID", "year"],
    how="left"
)

In [310]:
model_data_late = recent_teams.dropna(
    subset=[
        "previous_W",
        "previous_L",
        "previous_GF",
        "previous_GA",
        "previous_Pts",
        "previous_player_G",
        "previous_player_A",
        "previous_player_Pts",
        "previous_player_SOG",
        "previous_goalie_W",
        "previous_goalie_L",
        "previous_goalie_SHO",
        "previous_goalie_GA",
        "previous_goalie_SA",
        "previous_avg_win_rate_vs_opponent",
        "previous_std_win_rate_vs_opponent",
        "previous_young_player_ratio",
        "previous_veteran_player_ratio",
        "previous_forward_ratio",
        "previous_defense_ratio",
        "previous_goalie_ratio",
        "previous_late_season_win_rate"
    ]
)

model_data_late.shape

(373, 71)

In [311]:
features_late = [
    "previous_W",
    "previous_L",
    "previous_GF",
    "previous_GA",
    "previous_Pts",
    "previous_player_G",
    "previous_player_A",
    "previous_player_Pts",
    "previous_player_SOG",
    "previous_goalie_W",
    "previous_goalie_L",
    "previous_goalie_SHO",
    "previous_goalie_GA",
    "previous_goalie_SA",
    "previous_avg_win_rate_vs_opponent",
    "previous_std_win_rate_vs_opponent",
    "previous_young_player_ratio",
    "previous_veteran_player_ratio",
    "previous_forward_ratio",
    "previous_defense_ratio",
    "previous_goalie_ratio",
    "previous_late_season_win_rate"
]

In [312]:
train_late = model_data_late[
    model_data_late["year"] <= 2006
]

test_late = model_data_late[
    model_data_late["year"] >= 2007
]

X_train_late = train_late[features_late]
y_train_late = train_late["Pts"]

X_test_late = test_late[features_late]
y_test_late = test_late["Pts"]

late_scaler = StandardScaler()

X_train_late_scaled = late_scaler.fit_transform(
    X_train_late
)

X_test_late_scaled = late_scaler.transform(
    X_test_late
)

In [313]:
svr_late_model = SVR(
    kernel="rbf",
    C=1,
    epsilon=0.1
)

svr_late_model.fit(
    X_train_late_scaled,
    y_train_late
)

y_pred_late = svr_late_model.predict(
    X_test_late_scaled
)

In [314]:
late_rank_error, late_rank_corr = evaluate_ranking(
    test_late,
    y_pred_late
)

late_rank_error, late_rank_corr

(np.float64(6.311111111111111), np.float64(0.5207569828058112))

### Late-Season Form Evaluation

Adding previous late-season win rate did not improve ranking performance.

The model achieved:

- Mean Rank Error: 6.31 positions
- Rank Correlation: 0.52

Both metrics are slightly worse than the previous best model.

Therefore, `previous_late_season_win_rate` is not retained in the current best feature set.

In [315]:
features_best_current = [
    "previous_W",
    "previous_L",
    "previous_GF",
    "previous_GA",
    "previous_Pts",
    "previous_player_G",
    "previous_player_A",
    "previous_player_Pts",
    "previous_player_SOG",
    "previous_goalie_W",
    "previous_goalie_L",
    "previous_goalie_SHO",
    "previous_goalie_GA",
    "previous_goalie_SA",
    "previous_avg_win_rate_vs_opponent",
    "previous_std_win_rate_vs_opponent",
    "previous_young_player_ratio",
    "previous_veteran_player_ratio",
    "previous_forward_ratio",
    "previous_defense_ratio",
    "previous_goalie_ratio"
]

In [316]:
from sklearn.ensemble import RandomForestRegressor

# Use only the training period
feature_selection_data = model_data_position[
    model_data_position["year"] <= 2006
]

X_fs = feature_selection_data[features_best_current]
y_fs = feature_selection_data["Pts"]

rf_feature_selector = RandomForestRegressor(
    n_estimators=300,
    random_state=42
)

rf_feature_selector.fit(X_fs, y_fs)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease o

In [317]:
feature_importance = pd.DataFrame({
    "Feature": features_best_current,
    "Importance": rf_feature_selector.feature_importances_
}).sort_values(
    "Importance",
    ascending=False
)

feature_importance

,Feature,Importance
4,previous_Pts,0.161723
1,previous_L,0.101882
8,previous_player_SOG,0.084095
13,previous_goalie_SA,0.082933
10,previous_goalie_L,0.056910
17,previous_veteran_player_ratio,0.049601
3,previous_GA,0.047876
14,previous_avg_win_rate_vs_opponent,0.047820
15,previous_std_win_rate_vs_opponent,0.044077
20,previous_goalie_ratio,0.042460


In [318]:
top_5_features = feature_importance.head(5)["Feature"].tolist()
top_10_features = feature_importance.head(10)["Feature"].tolist()
top_15_features = feature_importance.head(15)["Feature"].tolist()
all_features = features_best_current

In [319]:
def test_feature_subset(feature_list):
    train_subset = model_data_position[
        model_data_position["year"] <= 2006
    ]

    test_subset = model_data_position[
        model_data_position["year"] >= 2007
    ]

    X_train_subset = train_subset[feature_list]
    y_train_subset = train_subset["Pts"]

    X_test_subset = test_subset[feature_list]
    y_test_subset = test_subset["Pts"]

    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train_subset)
    X_test_scaled = scaler.transform(X_test_subset)

    model = SVR(
        kernel="rbf",
        C=1,
        epsilon=0.1
    )

    model.fit(X_train_scaled, y_train_subset)

    predictions = model.predict(X_test_scaled)

    rank_error, rank_corr = evaluate_ranking(
        test_subset,
        predictions
    )

    return rank_error, rank_corr

In [320]:
feature_subset_results = []

for name, features in [
    ("Top 5", top_5_features),
    ("Top 10", top_10_features),
    ("Top 15", top_15_features),
    ("All Features", all_features)
]:
    rank_error, rank_corr = test_feature_subset(features)

    feature_subset_results.append({
        "Feature Set": name,
        "Number of Features": len(features),
        "Rank Error": rank_error,
        "Rank Correlation": rank_corr
    })

feature_subset_comparison = pd.DataFrame(
    feature_subset_results
)

feature_subset_comparison.sort_values("Rank Error")

,Feature Set,Number of Features,Rank Error,Rank Correlation
2,Top 15,15,6.288889,0.527165
3,All Features,21,6.288889,0.530443
1,Top 10,10,6.400000,0.529102
0,Top 5,5,6.488889,0.531039


### Feature Selection Evaluation

Different feature subsets were tested using the SVR model.

The Top 15 and full feature sets achieved the same mean rank error of approximately 6.29 positions.

However, the full feature set obtained a slightly higher rank correlation.

Therefore, the full 21-feature set is retained as the current best configuration.